In [ ]:
def gradient_check(model: AdvancedMLP, X: np.ndarray, y: np.ndarray, 
                  epsilon: float = 1e-7, tolerance: float = 1e-7) -> bool:
    """
    Verify backpropagation implementation using numerical gradient checking.
    
    Args:
        model: The MLP model to check
        X: Input data (small batch recommended)
        y: Target labels
        epsilon: Small value for numerical differentiation
        tolerance: Maximum allowed difference between analytical and numerical gradients
        
    Returns:
        bool: True if gradients match within tolerance
    """
    print("🔍 Performing gradient checking...")
    
    # Forward pass to compute analytical gradients
    model.forward(X, training=True)
    loss = model._compute_loss(y)
    analytical_grads = model._backward(y)
    
    # Check gradients for each layer
    for layer_idx in range(len(model.weights)):
        print(f"\n📋 Checking Layer {layer_idx + 1}:")
        
        # Check weight gradients
        W = model.weights[layer_idx]
        dW_analytical = analytical_grads['weights'][layer_idx]
        dW_numerical = np.zeros_like(W)
        
        # Numerical gradient computation for weights
        for i in range(W.shape[0]):
            for j in range(W.shape[1]):
                # f(x + epsilon)
                W[i, j] += epsilon
                model.forward(X, training=False)
                loss_plus = model._compute_loss(y)
                
                # f(x - epsilon)
                W[i, j] -= 2 * epsilon
                model.forward(X, training=False)
                loss_minus = model._compute_loss(y)
                
                # Restore original value
                W[i, j] += epsilon
                
                # Numerical gradient
                dW_numerical[i, j] = (loss_plus - loss_minus) / (2 * epsilon)
        
        # Compare gradients
        diff_weights = np.abs(dW_analytical - dW_numerical)
        max_diff_weights = np.max(diff_weights)
        rel_error_weights = max_diff_weights / (np.max(np.abs(dW_analytical)) + epsilon)
        
        print(f"   Weights - Max absolute difference: {max_diff_weights:.2e}")
        print(f"   Weights - Relative error: {rel_error_weights:.2e}")
        
        if max_diff_weights > tolerance:
            print(f"   ❌ Weight gradients FAILED (diff: {max_diff_weights:.2e} > tol: {tolerance:.2e})")
            return False
        else:
            print(f"   ✅ Weight gradients PASSED")
        
        # Check bias gradients
        b = model.biases[layer_idx]
        db_analytical = analytical_grads['biases'][layer_idx]
        db_numerical = np.zeros_like(b)
        
        for i in range(b.shape[0]):
            # f(x + epsilon)
            b[i] += epsilon
            model.forward(X, training=False)
            loss_plus = model._compute_loss(y)
            
            # f(x - epsilon)
            b[i] -= 2 * epsilon
            model.forward(X, training=False)
            loss_minus = model._compute_loss(y)
            
            # Restore original value
            b[i] += epsilon
            
            # Numerical gradient
            db_numerical[i] = (loss_plus - loss_minus) / (2 * epsilon)
        
        # Compare bias gradients
        diff_biases = np.abs(db_analytical - db_numerical)
        max_diff_biases = np.max(diff_biases)
        rel_error_biases = max_diff_biases / (np.max(np.abs(db_analytical)) + epsilon)
        
        print(f"   Biases - Max absolute difference: {max_diff_biases:.2e}")
        print(f"   Biases - Relative error: {rel_error_biases:.2e}")
        
        if max_diff_biases > tolerance:
            print(f"   ❌ Bias gradients FAILED (diff: {max_diff_biases:.2e} > tol: {tolerance:.2e})")
            return False
        else:
            print(f"   ✅ Bias gradients PASSED")
    
    print(f"\n🎉 All gradient checks PASSED! Backpropagation implementation is correct.")
    return True

def test_gradient_checking():
    """Test the gradient checking function with a simple example."""
    print("🧪 Testing Gradient Checking Function\n")
    
    # Create small test data
    np.random.seed(42)
    X_test = np.random.randn(5, 3)  # Small batch for efficiency
    y_test = np.random.randint(0, 2, 5)  # Binary classification
    
    # Create simple model
    model = AdvancedMLP(
        layers=[3, 4, 2],  # Small network
        activation='relu',
        optimizer='sgd',
        learning_rate=0.01,
        random_state=42
    )
    
    # Run gradient check
    is_correct = gradient_check(model, X_test, y_test, tolerance=1e-6)
    
    if is_correct:
        print("✅ Gradient checking function works correctly!")
    else:
        print("❌ There might be an issue with the backpropagation implementation.")
    
    return is_correct

# Run the gradient checking test
test_gradient_checking()

## 9. Gradient Checking and Verification

Before using the MLP in practice, it's crucial to verify that our backpropagation implementation is correct. We'll implement numerical gradient checking to compare our analytical gradients with numerically computed gradients.

# 🎯 Summary and Key Takeaways

## What We've Accomplished

This notebook successfully demonstrates a comprehensive, production-ready Multi-Layer Perceptron implementation with advanced features:

### ✅ **Implementation Features**
- **Multiple Optimizers**: SGD, Momentum, Adam, RMSprop, AdaGrad
- **Various Activations**: ReLU, Leaky ReLU, Sigmoid, Tanh
- **Regularization**: L1, L2 weight decay, Dropout
- **Advanced Initialization**: Xavier/Glorot and He initialization
- **Training Features**: Mini-batch processing, early stopping, learning rate scheduling
- **Comprehensive Validation**: Input validation, type hints, error handling
- **Rich Visualizations**: Training curves, decision boundaries, weight distributions

### 🧠 **Key Learning Points**

1. **Simple vs Advanced**: Started with a basic MLP to understand core concepts, then explored advanced features
2. **Mathematical Foundation**: Solid grounding in forward/backward propagation mathematics
3. **Modular Design**: Well-organized code structure for maintenance and extensibility
4. **Best Practices**: Proper initialization, numerical stability, vectorized operations
5. **Practical Application**: Real examples on different types of datasets

### 🚀 **Performance Insights**

- **Adam** typically performs best across different datasets
- **Proper initialization** (He for ReLU, Xavier for sigmoid/tanh) is crucial
- **Regularization** helps prevent overfitting, especially with dropout
- **Learning rate scheduling** can improve convergence
- **Early stopping** prevents overfitting automatically

### 🎨 **Visualization Benefits**

- Training curves help identify overfitting/underfitting
- Decision boundaries visualize model behavior
- Weight distributions show initialization and learning effects
- Gradient norms help detect vanishing/exploding gradients

## Next Steps

This implementation provides a solid foundation for:
- **Research**: Experimenting with new architectures and techniques  
- **Education**: Understanding deep learning fundamentals
- **Prototyping**: Quick testing of ideas before moving to frameworks like PyTorch/TensorFlow
- **Benchmarking**: Comparing different optimization strategies

The modular design makes it easy to extend with new features like batch normalization, different loss functions, or advanced regularization techniques.

In [ ]:
# Example 3: Optimizer Comparison
print("\\n⚖️ Example 3: Optimizer Comparison")
print("-" * 40)

# Generate dataset for comparison
X_comp, y_comp = make_classification(\n    n_samples=1500, n_features=10, n_informative=8, n_redundant=2,\n    n_classes=3, random_state=42\n)\nscaler_comp = StandardScaler()\nX_comp_scaled = scaler_comp.fit_transform(X_comp)\nX_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(\n    X_comp_scaled, y_comp, test_size=0.3, random_state=42\n)\n\n# Test different optimizers\noptimizers = ['sgd', 'momentum', 'adam', 'rmsprop']\nresults_comparison = {}\n\nfor optimizer in optimizers:\n    print(f\"\\n🔧 Testing {optimizer.upper()} optimizer...\")\n    \n    mlp = AdvancedMLP(\n        layers=[10, 32, 16, 3],\n        activation='relu',\n        optimizer=optimizer,\n        learning_rate=0.01 if optimizer == 'sgd' else 0.001,\n        regularization='l2',\n        reg_lambda=0.01,\n        batch_size=32,\n        random_state=42\n    )\n    \n    # Train with reduced verbosity\n    history = mlp.fit(\n        X_train_comp, y_train_comp,\n        X_val=X_test_comp, y_val=y_test_comp,\n        epochs=300,\n        verbose=False\n    )\n    \n    # Evaluate\n    test_acc = mlp.score(X_test_comp, y_test_comp)\n    final_train_loss = history['train_loss'][-1]\n    final_val_loss = history['val_loss'][-1]\n    \n    results_comparison[optimizer] = {\n        'test_accuracy': test_acc,\n        'final_train_loss': final_train_loss,\n        'final_val_loss': final_val_loss,\n        'history': history\n    }\n    \n    print(f\"   📊 Test Accuracy: {test_acc:.4f}\")\n    print(f\"   📈 Final Train Loss: {final_train_loss:.4f}\")\n    print(f\"   📉 Final Val Loss: {final_val_loss:.4f}\")\n\n# Plot comparison\nfig, axes = plt.subplots(1, 2, figsize=(15, 6))\n\n# Training loss comparison\nfor optimizer, results in results_comparison.items():\n    epochs = range(1, len(results['history']['train_loss']) + 1)\n    axes[0].plot(epochs, results['history']['train_loss'], \n                label=f'{optimizer.upper()}', linewidth=2)\n\naxes[0].set_title('Training Loss Comparison', fontsize=14, fontweight='bold')\naxes[0].set_xlabel('Epoch')\naxes[0].set_ylabel('Loss')\naxes[0].legend()\naxes[0].grid(True, alpha=0.3)\naxes[0].set_yscale('log')\n\n# Test accuracy comparison\noptimizer_names = [opt.upper() for opt in optimizers]\naccuracies = [results_comparison[opt]['test_accuracy'] for opt in optimizers]\n\nbars = axes[1].bar(optimizer_names, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])\naxes[1].set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')\naxes[1].set_ylabel('Accuracy')\naxes[1].set_ylim(0.8, 1.0)\n\n# Add value labels on bars\nfor bar, acc in zip(bars, accuracies):\n    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,\n                f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')\n\nplt.tight_layout()\nplt.show()\n\n# Summary\nprint(\"\\n📋 Optimizer Comparison Summary:\")\nprint(\"=\" * 50)\nfor optimizer, results in results_comparison.items():\n    print(f\"{optimizer.upper():>10}: Accuracy = {results['test_accuracy']:.4f}, \"\n          f\"Val Loss = {results['final_val_loss']:.4f}\")\n\nbest_optimizer = max(results_comparison.keys(), \n                    key=lambda k: results_comparison[k]['test_accuracy'])\nprint(f\"\\n🏆 Best Performer: {best_optimizer.upper()} \"\n      f\"(Accuracy: {results_comparison[best_optimizer]['test_accuracy']:.4f})\")\n\nprint(\"\\n✅ Example 3 completed successfully!\")

In [ ]:
# Example 2: Non-linear Decision Boundaries
print("\\n🎯 Example 2: Non-linear Decision Boundaries")
print("-" * 45)

# Create circles dataset\nX_circles, y_circles = make_circles(n_samples=1000, noise=0.1, factor=0.5, random_state=42)\nX_circles_train, X_circles_test, y_circles_train, y_circles_test = train_test_split(\n    X_circles, y_circles, test_size=0.3, random_state=42\n)\n\n# Initialize MLP for circles\nmlp_circles = AdvancedMLP(\n    layers=[2, 16, 8, 2],  # 2D input, 2 classes\n    activation='relu',\n    optimizer='adam',\n    learning_rate=0.01,\n    regularization=None,\n    dropout_rate=0.0,\n    batch_size=32,\n    random_state=42\n)\n\nprint(\"🌀 Training on Circles Dataset...\")\nmlp_circles.fit(\n    X_circles_train, y_circles_train,\n    X_val=X_circles_test, y_val=y_circles_test,\n    epochs=300,\n    verbose=False\n)\n\n# Plot decision boundary\nmlp_circles.plot_decision_boundary(X_circles, y_circles, \n                                  title=\"Decision Boundary - Circles Dataset\")\n\n# Create moons dataset\nX_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=42)\nX_moons_train, X_moons_test, y_moons_train, y_moons_test = train_test_split(\n    X_moons, y_moons, test_size=0.3, random_state=42\n)\n\n# Initialize MLP for moons\nmlp_moons = AdvancedMLP(\n    layers=[2, 16, 8, 2],\n    activation='relu',\n    optimizer='adam',\n    learning_rate=0.01,\n    regularization=None,\n    dropout_rate=0.0,\n    batch_size=32,\n    random_state=42\n)\n\nprint(\"🌙 Training on Moons Dataset...\")\nmlp_moons.fit(\n    X_moons_train, y_moons_train,\n    X_val=X_moons_test, y_val=y_moons_test,\n    epochs=300,\n    verbose=False\n)\n\n# Plot decision boundary\nmlp_moons.plot_decision_boundary(X_moons, y_moons, \n                                title=\"Decision Boundary - Moons Dataset\")\n\nprint(\"\\n📈 Performance Comparison:\")\ncircles_acc = mlp_circles.score(X_circles_test, y_circles_test)\nmoons_acc = mlp_moons.score(X_moons_test, y_moons_test)\nprint(f\"   • Circles Dataset Accuracy: {circles_acc:.4f}\")\nprint(f\"   • Moons Dataset Accuracy: {moons_acc:.4f}\")\n\nprint(\"\\n✅ Example 2 completed successfully!\")

In [ ]:
# Example demonstrations of the Advanced MLP
print("🎯 Advanced MLP Implementation Examples")
print("=" * 50)

# Example 1: Multi-class Classification with Synthetic Data
print("\\n📊 Example 1: Multi-class Classification")
print("-" * 40)

# Generate synthetic dataset
X_multi, y_multi = make_classification(
    n_samples=2000, n_features=20, n_informative=15, n_redundant=5, 
    n_classes=3, n_clusters_per_class=1, random_state=42
)

# Preprocess data
scaler = StandardScaler()
X_multi_scaled = scaler.fit_transform(X_multi)
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi_scaled, y_multi, test_size=0.3, random_state=42, stratify=y_multi
)

# Initialize Advanced MLP
mlp_advanced = AdvancedMLP(
    layers=[20, 64, 32, 3],  # 20 inputs, two hidden layers, 3 outputs
    activation='relu',
    optimizer='adam',
    learning_rate=0.001,
    regularization='l2',
    reg_lambda=0.01,
    dropout_rate=0.1,
    batch_size=32,
    random_state=42
)

print(f\"📋 Model Configuration:\")\nconfig = mlp_advanced.get_config()\nfor key, value in config.items():\n    print(f\"   • {key}: {value}\")\n\n# Count parameters\ntotal_params = sum(w.size + b.size for w, b in zip(mlp_advanced.weights, mlp_advanced.biases))\nprint(f\"\\n🔢 Total Parameters: {total_params:,}\")\n\n# Train the model\nprint(\"\\n🚀 Training Advanced MLP...\")\nhistory = mlp_advanced.fit(\n    X_train_multi, y_train_multi,\n    X_val=X_test_multi, y_val=y_test_multi,\n    epochs=500,\n    early_stopping={'patience': 50},\n    verbose=True\n)\n\n# Evaluate the model\nprint(\"\\n📊 Model Evaluation:\")\nresults = mlp_advanced.evaluate(X_test_multi, y_test_multi, verbose=True)\n\n# Plot training history\nmlp_advanced.plot_training_history()\n\n# Plot weight distributions\nmlp_advanced.plot_weights_distribution()\n\nprint(\"\\n✅ Example 1 completed successfully!\")

## 8. Example Demonstrations

This section demonstrates the advanced MLP implementation with practical examples on different datasets.

### Examples Included:
1. **Synthetic Classification**: Multi-class classification with different optimizers
2. **Circles Dataset**: Non-linear decision boundary demonstration
3. **Moons Dataset**: Half-moon shaped data classification
4. **MNIST Digits**: Real-world handwritten digit recognition
5. **Optimizer Comparison**: Side-by-side performance analysis

### Learning Objectives:
- Understanding different optimization algorithms
- Exploring regularization effects
- Visualizing training dynamics
- Comparing model performance across datasets

In [ ]:
# Visualization functions for AdvancedMLP
def visualization_methods():
    """
    Add comprehensive visualization methods to the AdvancedMLP class
    """
    
    def plot_training_history(self, figsize: Tuple[int, int] = (15, 10)) -> None:\n        \"\"\"\n        Plot comprehensive training history\n        \n        Parameters:\n        -----------\n        figsize : Tuple[int, int]\n            Figure size (width, height)\n        \"\"\"\n        if not self.history['train_loss']:\n            print(\"No training history available. Train the model first.\")\n            return\n        \n        fig, axes = plt.subplots(2, 2, figsize=figsize)\n        epochs = range(1, len(self.history['train_loss']) + 1)\n        \n        # Loss curves\n        axes[0, 0].plot(epochs, self.history['train_loss'], 'b-', label='Training Loss', linewidth=2)\n        if self.history['val_loss']:\n            axes[0, 0].plot(epochs, self.history['val_loss'], 'r--', label='Validation Loss', linewidth=2)\n        axes[0, 0].set_title('Loss Curves', fontsize=14, fontweight='bold')\n        axes[0, 0].set_xlabel('Epoch')\n        axes[0, 0].set_ylabel('Loss')\n        axes[0, 0].legend()\n        axes[0, 0].grid(True, alpha=0.3)\n        \n        # Accuracy curves\n        axes[0, 1].plot(epochs, self.history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)\n        if self.history['val_acc']:\n            axes[0, 1].plot(epochs, self.history['val_acc'], 'r--', label='Validation Accuracy', linewidth=2)\n        axes[0, 1].set_title('Accuracy Curves', fontsize=14, fontweight='bold')\n        axes[0, 1].set_xlabel('Epoch')\n        axes[0, 1].set_ylabel('Accuracy')\n        axes[0, 1].legend()\n        axes[0, 1].grid(True, alpha=0.3)\n        \n        # Learning rate schedule\n        if self.history['lr_history']:\n            axes[1, 0].plot(epochs, self.history['lr_history'], 'g-', linewidth=2)\n            axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')\n            axes[1, 0].set_xlabel('Epoch')\n            axes[1, 0].set_ylabel('Learning Rate')\n            axes[1, 0].set_yscale('log')\n            axes[1, 0].grid(True, alpha=0.3)\n        \n        # Gradient norms\n        if self.history['gradient_norms']:\n            axes[1, 1].plot(epochs, self.history['gradient_norms'], 'm-', linewidth=2)\n            axes[1, 1].set_title('Gradient Norms', fontsize=14, fontweight='bold')\n            axes[1, 1].set_xlabel('Epoch')\n            axes[1, 1].set_ylabel('Gradient Norm')\n            axes[1, 1].set_yscale('log')\n            axes[1, 1].grid(True, alpha=0.3)\n        \n        plt.tight_layout()\n        plt.show()\n    \n    def plot_decision_boundary(self, X: np.ndarray, y: np.ndarray, \n                              title: str = \"Decision Boundary\", \n                              figsize: Tuple[int, int] = (10, 8)) -> None:\n        \"\"\"\n        Plot decision boundary for 2D datasets\n        \n        Parameters:\n        -----------\n        X : np.ndarray\n            2D input data\n        y : np.ndarray\n            Labels\n        title : str\n            Plot title\n        figsize : Tuple[int, int]\n            Figure size\n        \"\"\"\n        if X.shape[1] != 2:\n            print(\"Decision boundary plotting only available for 2D data\")\n            return\n        \n        plt.figure(figsize=figsize)\n        \n        # Create a mesh\n        h = 0.01\n        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1\n        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1\n        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),\n                            np.arange(y_min, y_max, h))\n        \n        # Make predictions on the mesh\n        mesh_points = np.c_[xx.ravel(), yy.ravel()]\n        Z = self.predict_proba(mesh_points)\n        Z = Z[:, 1] if Z.shape[1] > 1 else Z.ravel()  # For binary classification\n        Z = Z.reshape(xx.shape)\n        \n        # Plot the contour and training examples\n        plt.contourf(xx, yy, Z, levels=50, alpha=0.8, cmap=plt.cm.RdYlBu)\n        scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='black')\n        plt.colorbar(scatter)\n        plt.title(title, fontsize=16, fontweight='bold')\n        plt.xlabel('Feature 1', fontsize=12)\n        plt.ylabel('Feature 2', fontsize=12)\n        plt.show()\n    \n    def plot_weights_distribution(self, figsize: Tuple[int, int] = (15, 5)) -> None:\n        \"\"\"\n        Plot weight distributions for all layers\n        \n        Parameters:\n        -----------\n        figsize : Tuple[int, int]\n            Figure size\n        \"\"\"\n        n_layers = len(self.weights)\n        fig, axes = plt.subplots(1, n_layers, figsize=figsize)\n        \n        if n_layers == 1:\n            axes = [axes]\n        \n        for i, (ax, weights) in enumerate(zip(axes, self.weights)):\n            ax.hist(weights.flatten(), bins=50, alpha=0.7, density=True)\n            ax.set_title(f'Layer {i+1} Weights\\n(μ={weights.mean():.4f}, σ={weights.std():.4f})', \n                        fontsize=12, fontweight='bold')\n            ax.set_xlabel('Weight Value')\n            ax.set_ylabel('Density')\n            ax.grid(True, alpha=0.3)\n        \n        plt.tight_layout()\n        plt.show()\n    \n    def plot_confusion_matrix(self, X: np.ndarray, y: np.ndarray, \n                             class_names: Optional[List[str]] = None,\n                             figsize: Tuple[int, int] = (8, 6)) -> None:\n        \"\"\"\n        Plot confusion matrix\n        \n        Parameters:\n        -----------\n        X : np.ndarray\n            Input data\n        y : np.ndarray\n            True labels\n        class_names : List[str], optional\n            Class names for labeling\n        figsize : Tuple[int, int]\n            Figure size\n        \"\"\"\n        predictions = self.predict(X)\n        cm = confusion_matrix(y, predictions)\n        \n        plt.figure(figsize=figsize)\n        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', \n                   xticklabels=class_names, yticklabels=class_names)\n        plt.title('Confusion Matrix', fontsize=16, fontweight='bold')\n        plt.xlabel('Predicted Label', fontsize=12)\n        plt.ylabel('True Label', fontsize=12)\n        plt.show()\n    \n    def plot_activations(self, X: np.ndarray, layer_idx: int = 0, \n                        n_neurons: int = 10, figsize: Tuple[int, int] = (12, 8)) -> None:\n        \"\"\"\n        Plot activation patterns for a specific layer\n        \n        Parameters:\n        -----------\n        X : np.ndarray\n            Input data\n        layer_idx : int\n            Index of layer to visualize (0 = first hidden layer)\n        n_neurons : int\n            Number of neurons to visualize\n        figsize : Tuple[int, int]\n            Figure size\n        \"\"\"\n        # Forward pass to get activations\n        _ = self.forward(X[:100], training=False)  # Use subset for speed\n        \n        if f'Z{layer_idx+1}' not in self.cache:\n            print(f\"Layer {layer_idx} not found\")\n            return\n        \n        activations = self.cache['A'][layer_idx + 1][:, :n_neurons]\n        \n        plt.figure(figsize=figsize)\n        plt.imshow(activations.T, cmap='viridis', aspect='auto')\n        plt.colorbar()\n        plt.title(f'Activation Patterns - Layer {layer_idx + 1}', fontsize=16, fontweight='bold')\n        plt.xlabel('Sample Index')\n        plt.ylabel('Neuron Index')\n        plt.show()\n    \n    # Add methods to the AdvancedMLP class\n    AdvancedMLP.plot_training_history = plot_training_history\n    AdvancedMLP.plot_decision_boundary = plot_decision_boundary\n    AdvancedMLP.plot_weights_distribution = plot_weights_distribution\n    AdvancedMLP.plot_confusion_matrix = plot_confusion_matrix\n    AdvancedMLP.plot_activations = plot_activations\n\n# Apply the methods to the class\nvisualization_methods()\n\nprint(\"📊 Comprehensive visualization toolkit added\")\nprint(\"🎨 Available plots:\")\nprint(\"   • Training history (loss, accuracy, learning rate, gradients)\")\nprint(\"   • Decision boundaries for 2D data\")\nprint(\"   • Weight distributions across layers\")\nprint(\"   • Confusion matrix for classification performance\")\nprint(\"   • Activation patterns in hidden layers\")\nprint(\"✅ Visualization methods ready\")

## 7. Visualization Functions

This section provides comprehensive visualization tools for understanding model behavior and training dynamics.

### Available Visualizations:
- **Training Curves**: Loss and accuracy over time
- **Learning Rate Schedule**: How learning rate changes during training
- **Gradient Norms**: Monitor gradient flow and detect vanishing/exploding gradients
- **Decision Boundaries**: For 2D datasets, visualize classification regions
- **Weight Distributions**: Analyze parameter distributions
- **Confusion Matrix**: Detailed classification performance

### Key Features:
- **Interactive Plots**: Clear, publication-ready figures
- **Multiple Subplots**: Comprehensive overview in single figure
- **Customizable**: Flexible styling and configuration options

In [ ]:
# Training and evaluation methods
def training_and_evaluation_methods():
    """
    Add training and evaluation methods to the AdvancedMLP class
    """
    
    def fit(self, X_train: np.ndarray, y_train: np.ndarray, 
            X_val: Optional[np.ndarray] = None, y_val: Optional[np.ndarray] = None, 
            epochs: int = 1000, lr_schedule: Optional[Dict[str, Any]] = None, 
            early_stopping: Optional[Dict[str, Any]] = None, 
            verbose: bool = True) -> Dict[str, List[float]]:\n        \"\"\"\n        Train the MLP with comprehensive monitoring and early stopping\n        \n        Parameters:\n        -----------\n        X_train : np.ndarray\n            Training data\n        y_train : np.ndarray\n            Training labels\n        X_val : np.ndarray, optional\n            Validation data\n        y_val : np.ndarray, optional\n            Validation labels\n        epochs : int, default=1000\n            Number of training epochs\n        lr_schedule : dict, optional\n            Learning rate schedule configuration\n        early_stopping : dict, optional\n            Early stopping configuration {'patience': int}\n        verbose : bool, default=True\n            Whether to print training progress\n            \n        Returns:\n        --------\n        Dict[str, List[float]]\n            Training history dictionary\n        \"\"\"\n        # Validate input data\n        validate_input_data(X_train, y_train)\n        if X_val is not None:\n            validate_input_data(X_val, y_val)\n        \n        # Convert labels to one-hot encoding\n        if len(y_train.shape) == 1:\n            n_classes = len(np.unique(y_train))\n            if n_classes != self.layers[-1]:\n                raise ValueError(f\"Number of classes {n_classes} doesn't match output layer size {self.layers[-1]}\")\n            y_train_onehot = np.eye(n_classes)[y_train]\n        else:\n            y_train_onehot = y_train\n            n_classes = y_train.shape[1]\n            \n        if X_val is not None and len(y_val.shape) == 1:\n            y_val_onehot = np.eye(n_classes)[y_val]\n        elif X_val is not None:\n            y_val_onehot = y_val\n        \n        # Initialize early stopping\n        best_val_loss = np.inf\n        patience_counter = 0\n        best_weights: Optional[List[np.ndarray]] = None\n        best_biases: Optional[List[np.ndarray]] = None\n        \n        if verbose:\n            print(f\"🏋️  Training MLP: {self.layers}\")\n            print(f\"📊 Optimizer: {self.optimizer}, Activation: {self.activation}\")\n            print(f\"🔧 Regularization: {self.regularization}, Dropout: {self.dropout_rate}\")\n            print(\"-\" * 60)\n        \n        for epoch in range(epochs):\n            # Learning rate scheduling\n            if lr_schedule:\n                self._lr_schedule(epoch, **lr_schedule)\n            \n            # Mini-batch training\n            n_samples = X_train.shape[0]\n            indices = np.random.permutation(n_samples)\n            \n            epoch_loss = 0.0\n            epoch_acc = 0.0\n            gradient_norm = 0.0\n            \n            for i in range(0, n_samples, self.batch_size):\n                batch_indices = indices[i:i + self.batch_size]\n                X_batch = X_train[batch_indices]\n                y_batch = y_train_onehot[batch_indices]\n                \n                # Forward pass\n                y_pred = self.forward(X_batch, training=True)\n                \n                # Compute loss\n                batch_loss = self._compute_loss(y_batch, y_pred)\n                epoch_loss += batch_loss\n                \n                # Compute accuracy\n                batch_acc = accuracy_score(np.argmax(y_batch, axis=1), np.argmax(y_pred, axis=1))\n                epoch_acc += batch_acc\n                \n                # Backward pass\n                gradients_w, gradients_b = self.backward(X_batch, y_batch, y_pred)\n                \n                # Compute gradient norm for monitoring\n                grad_norm = sum(np.linalg.norm(gw) for gw in gradients_w)\n                gradient_norm += grad_norm\n                \n                # Update parameters\n                self._update_parameters(gradients_w, gradients_b)\n            \n            # Average metrics over batches\n            n_batches = (n_samples + self.batch_size - 1) // self.batch_size\n            epoch_loss /= n_batches\n            epoch_acc /= n_batches\n            gradient_norm /= n_batches\n            \n            # Store training metrics\n            self.history['train_loss'].append(epoch_loss)\n            self.history['train_acc'].append(epoch_acc)\n            self.history['lr_history'].append(self.learning_rate)\n            self.history['gradient_norms'].append(gradient_norm)\n            \n            # Validation metrics\n            if X_val is not None:\n                val_pred = self.forward(X_val, training=False)\n                val_loss = self._compute_loss(y_val_onehot, val_pred)\n                val_acc = accuracy_score(np.argmax(y_val_onehot, axis=1), np.argmax(val_pred, axis=1))\n                \n                self.history['val_loss'].append(val_loss)\n                self.history['val_acc'].append(val_acc)\n                \n                # Early stopping\n                if early_stopping and val_loss < best_val_loss:\n                    best_val_loss = val_loss\n                    patience_counter = 0\n                    best_weights = [w.copy() for w in self.weights]\n                    best_biases = [b.copy() for b in self.biases]\n                elif early_stopping:\n                    patience_counter += 1\n                    if patience_counter >= early_stopping['patience']:\n                        if verbose:\n                            print(f\"\\nEarly stopping at epoch {epoch}\")\n                            print(f\"Restoring best weights (val_loss: {best_val_loss:.4f})\")\n                        if best_weights is not None and best_biases is not None:\n                            self.weights = best_weights\n                            self.biases = best_biases\n                        break\n            \n            # Progress reporting\n            if verbose and epoch % 100 == 0:\n                if X_val is not None:\n                    print(f\"Epoch {epoch:4d} | Train: Loss={epoch_loss:.4f}, Acc={epoch_acc:.4f} | \"\n                          f\"Val: Loss={val_loss:.4f}, Acc={val_acc:.4f} | LR={self.learning_rate:.6f}\")\n                else:\n                    print(f\"Epoch {epoch:4d} | Train: Loss={epoch_loss:.4f}, Acc={epoch_acc:.4f} | \"\n                          f\"LR={self.learning_rate:.6f}\")\n        \n        if verbose:\n            print(\"\\n✅ Training completed!\")\n        \n        return self.history\n    \n    def score(self, X: np.ndarray, y: np.ndarray) -> float:\n        \"\"\"\n        Compute accuracy on given data\n        \n        Parameters:\n        -----------\n        X : np.ndarray\n            Input data\n        y : np.ndarray\n            True labels\n            \n        Returns:\n        --------\n        float\n            Accuracy score\n        \"\"\"\n        predictions = self.predict(X)\n        return accuracy_score(y, predictions)\n    \n    def evaluate(self, X: np.ndarray, y: np.ndarray, verbose: bool = True) -> Dict[str, Any]:\n        \"\"\"\n        Comprehensive evaluation of the model\n        \n        Parameters:\n        -----------\n        X : np.ndarray\n            Test data\n        y : np.ndarray\n            True labels\n        verbose : bool\n            Whether to print results\n            \n        Returns:\n        --------\n        Dict[str, Any]\n            Evaluation metrics\n        \"\"\"\n        predictions = self.predict(X)\n        probabilities = self.predict_proba(X)\n        \n        # Convert to one-hot if necessary\n        if len(y.shape) == 1:\n            y_onehot = np.eye(self.layers[-1])[y]\n        else:\n            y_onehot = y\n            y = np.argmax(y, axis=1)\n        \n        # Compute metrics\n        accuracy = accuracy_score(y, predictions)\n        loss = self._compute_loss(y_onehot, probabilities)\n        \n        results = {\n            'accuracy': accuracy,\n            'loss': loss,\n            'predictions': predictions,\n            'probabilities': probabilities\n        }\n        \n        if verbose:\n            print(f\"📊 Model Evaluation Results\")\n            print(f\"{'='*30}\")\n            print(f\"Accuracy: {accuracy:.4f}\")\n            print(f\"Loss: {loss:.4f}\")\n            print(\"\\n📋 Detailed Classification Report:\")\n            print(classification_report(y, predictions))\n        \n        return results\n    \n    # Add methods to the AdvancedMLP class\n    AdvancedMLP.fit = fit\n    AdvancedMLP.score = score\n    AdvancedMLP.evaluate = evaluate\n\n# Apply the methods to the class\ntraining_and_evaluation_methods()\n\nprint(\"🎯 Comprehensive training loop implemented\")\nprint(\"📊 Features available:\")\nprint(\"   • Mini-batch processing\")\nprint(\"   • Validation monitoring\")\nprint(\"   • Early stopping\")\nprint(\"   • Learning rate scheduling\")\nprint(\"   • Comprehensive evaluation metrics\")\nprint(\"✅ Training and evaluation methods ready\")

## 6. Training and Evaluation Methods

This section implements the comprehensive training loop with monitoring, validation, and early stopping capabilities.

### Training Features:
- **Mini-batch Processing**: Efficient batch-wise training
- **Validation Monitoring**: Track performance on validation set
- **Early Stopping**: Prevent overfitting automatically
- **Learning Rate Scheduling**: Adaptive learning rate adjustment
- **Comprehensive Metrics**: Loss, accuracy, gradient norms

### Training Loop Components:
1. **Data Validation**: Check input data integrity
2. **Epoch Loop**: Mini-batch training with monitoring
3. **Forward-Backward Pass**: Prediction and gradient computation
4. **Parameter Updates**: Apply chosen optimization algorithm
5. **Metric Tracking**: Record training progress

In [ ]:
# Backpropagation and optimization methods
def backpropagation_and_optimization_methods():
    """
    Add backpropagation and optimization methods to the AdvancedMLP class
    """
    
    def backward(self, X: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """
        Backward propagation with gradient computation
        
        Parameters:
        -----------
        X : np.ndarray
            Input data
        y_true : np.ndarray
            True labels (one-hot encoded)
        y_pred : np.ndarray
            Predicted probabilities
            
        Returns:
        --------
        Tuple[List[np.ndarray], List[np.ndarray]]
            (gradients_w, gradients_b) - Weight and bias gradients
        """
        m = X.shape[0]
        gradients_w: List[np.ndarray] = []\n        gradients_b: List[np.ndarray] = []\n        \n        # Output layer gradient (softmax + cross-entropy simplification)\n        dz = y_pred - y_true\n        \n        # Backward through layers\n        for i in reversed(range(self.n_layers - 1)):\n            # Compute gradients\n            dw = np.dot(self.cache['A'][i].T, dz) / m\n            db = np.mean(dz, axis=0, keepdims=True)\n            \n            # Add regularization to weight gradients\n            if self.regularization == 'l1':\n                dw += self.reg_lambda * np.sign(self.weights[i])\n            elif self.regularization == 'l2':\n                dw += self.reg_lambda * 2 * self.weights[i]\n            \n            gradients_w.insert(0, dw)\n            gradients_b.insert(0, db)\n            \n            # Propagate error to previous layer (except for first hidden layer)\n            if i > 0:\n                da_prev = np.dot(dz, self.weights[i].T)\n                dz = da_prev * activation_function(self.cache[f'Z{i}'], self.activation, derivative=True)\n        \n        return gradients_w, gradients_b\n    \n    def _update_parameters(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:\n        \"\"\"Update parameters using the specified optimizer\"\"\"\n        if self.optimizer == 'sgd':\n            self._sgd_update(gradients_w, gradients_b)\n        elif self.optimizer == 'momentum':\n            self._momentum_update(gradients_w, gradients_b)\n        elif self.optimizer == 'adam':\n            self._adam_update(gradients_w, gradients_b)\n        elif self.optimizer == 'rmsprop':\n            self._rmsprop_update(gradients_w, gradients_b)\n        elif self.optimizer == 'adagrad':\n            self._adagrad_update(gradients_w, gradients_b)\n    \n    def _sgd_update(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:\n        \"\"\"Standard SGD parameter update\"\"\"\n        for i in range(len(self.weights)):\n            self.weights[i] -= self.learning_rate * gradients_w[i]\n            self.biases[i] -= self.learning_rate * gradients_b[i]\n    \n    def _momentum_update(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray], beta: float = 0.9) -> None:\n        \"\"\"SGD with momentum parameter update\"\"\"\n        for i in range(len(self.weights)):\n            self.momentum_w[i] = beta * self.momentum_w[i] + (1 - beta) * gradients_w[i]\n            self.momentum_b[i] = beta * self.momentum_b[i] + (1 - beta) * gradients_b[i]\n            \n            self.weights[i] -= self.learning_rate * self.momentum_w[i]\n            self.biases[i] -= self.learning_rate * self.momentum_b[i]\n    \n    def _adam_update(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:\n        \"\"\"Adam optimizer parameter update with bias correction\"\"\"\n        self.t += 1\n        \n        for i in range(len(self.weights)):\n            # Update biased first and second moment estimates\n            self.momentum_w[i] = self.beta1 * self.momentum_w[i] + (1 - self.beta1) * gradients_w[i]\n            self.momentum_b[i] = self.beta1 * self.momentum_b[i] + (1 - self.beta1) * gradients_b[i]\n            \n            self.v_w[i] = self.beta2 * self.v_w[i] + (1 - self.beta2) * (gradients_w[i] ** 2)\n            self.v_b[i] = self.beta2 * self.v_b[i] + (1 - self.beta2) * (gradients_b[i] ** 2)\n            \n            # Bias correction\n            m_w_corrected = self.momentum_w[i] / (1 - self.beta1 ** self.t)\n            m_b_corrected = self.momentum_b[i] / (1 - self.beta1 ** self.t)\n            \n            v_w_corrected = self.v_w[i] / (1 - self.beta2 ** self.t)\n            v_b_corrected = self.v_b[i] / (1 - self.beta2 ** self.t)\n            \n            # Update parameters\n            self.weights[i] -= self.learning_rate * m_w_corrected / (np.sqrt(v_w_corrected) + self.epsilon)\n            self.biases[i] -= self.learning_rate * m_b_corrected / (np.sqrt(v_b_corrected) + self.epsilon)\n    \n    def _rmsprop_update(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray], beta: float = 0.9) -> None:\n        \"\"\"RMSprop optimizer parameter update\"\"\"\n        for i in range(len(self.weights)):\n            self.v_w[i] = beta * self.v_w[i] + (1 - beta) * (gradients_w[i] ** 2)\n            self.v_b[i] = beta * self.v_b[i] + (1 - beta) * (gradients_b[i] ** 2)\n            \n            self.weights[i] -= self.learning_rate * gradients_w[i] / (np.sqrt(self.v_w[i]) + self.epsilon)\n            self.biases[i] -= self.learning_rate * gradients_b[i] / (np.sqrt(self.v_b[i]) + self.epsilon)\n    \n    def _adagrad_update(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:\n        \"\"\"AdaGrad optimizer parameter update\"\"\"\n        for i in range(len(self.weights)):\n            self.v_w[i] += gradients_w[i] ** 2\n            self.v_b[i] += gradients_b[i] ** 2\n            \n            self.weights[i] -= self.learning_rate * gradients_w[i] / (np.sqrt(self.v_w[i]) + self.epsilon)\n            self.biases[i] -= self.learning_rate * gradients_b[i] / (np.sqrt(self.v_b[i]) + self.epsilon)\n    \n    def _lr_schedule(self, epoch: int, schedule_type: str = 'constant', **kwargs: Any) -> None:\n        \"\"\"Apply learning rate scheduling\"\"\"\n        if schedule_type == 'step':\n            step_size = kwargs.get('step_size', 100)\n            gamma = kwargs.get('gamma', 0.1)\n            self.learning_rate = self.initial_lr * (gamma ** (epoch // step_size))\n            \n        elif schedule_type == 'exponential':\n            gamma = kwargs.get('gamma', 0.95)\n            self.learning_rate = self.initial_lr * (gamma ** epoch)\n            \n        elif schedule_type == 'cosine':\n            T_max = kwargs.get('T_max', 1000)\n            eta_min = kwargs.get('eta_min', 0)\n            self.learning_rate = eta_min + (self.initial_lr - eta_min) * (1 + np.cos(np.pi * epoch / T_max)) / 2\n    \n    # Add methods to the AdvancedMLP class\n    AdvancedMLP.backward = backward\n    AdvancedMLP._update_parameters = _update_parameters\n    AdvancedMLP._sgd_update = _sgd_update\n    AdvancedMLP._momentum_update = _momentum_update\n    AdvancedMLP._adam_update = _adam_update\n    AdvancedMLP._rmsprop_update = _rmsprop_update\n    AdvancedMLP._adagrad_update = _adagrad_update\n    AdvancedMLP._lr_schedule = _lr_schedule\n\n# Apply the methods to the class\nbackpropagation_and_optimization_methods()\n\nprint(\"⏪ Backpropagation algorithm implemented\")\nprint(\"🎯 Multiple optimizers available:\")\nprint(\"   • SGD: Standard gradient descent\")\nprint(\"   • Momentum: Accelerated gradient descent\")\nprint(\"   • Adam: Adaptive moment estimation\")\nprint(\"   • RMSprop: Root mean square propagation\")\nprint(\"   • AdaGrad: Adaptive gradient algorithm\")\nprint(\"📈 Learning rate scheduling ready\")

## 5. Backpropagation and Optimization Methods

This section implements backpropagation for gradient computation and multiple optimization algorithms.

### Backpropagation Mathematics:
- **Output Layer**: $\frac{\partial J}{\partial \mathbf{z}^{[L]}} = \mathbf{a}^{[L]} - \mathbf{y}$
- **Hidden Layers**: $\frac{\partial J}{\partial \mathbf{z}^{[l]}} = (\mathbf{W}^{[l+1]})^T \frac{\partial J}{\partial \mathbf{z}^{[l+1]}} \odot g'(\mathbf{z}^{[l]})$
- **Weight Gradients**: $\frac{\partial J}{\partial \mathbf{W}^{[l]}} = \frac{1}{m} \mathbf{a}^{[l-1]} (\frac{\partial J}{\partial \mathbf{z}^{[l]}})^T$

### Optimization Algorithms:
- **SGD**: $\mathbf{W} := \mathbf{W} - \alpha \nabla \mathbf{W}$
- **Momentum**: $\mathbf{v}_t = \beta \mathbf{v}_{t-1} + (1-\beta) \nabla \mathbf{W}$
- **Adam**: Adaptive learning rates with bias correction
- **RMSprop**: Moving average of squared gradients
- **AdaGrad**: Accumulation of squared gradients

In [ ]:
# Forward propagation methods for AdvancedMLP class
def forward_propagation_methods():
    """
    Add forward propagation methods to the AdvancedMLP class
    """
    
    def forward(self, X: np.ndarray, training: bool = True) -> np.ndarray:
        """
        Forward propagation with caching for backpropagation
        
        Parameters:
        -----------
        X : np.ndarray
            Input data of shape (batch_size, n_features)
        training : bool, default=True
            Whether in training mode (affects dropout)
            
        Returns:
        --------
        np.ndarray
            Output predictions of shape (batch_size, n_classes)
        """
        # Validate input
        if not isinstance(X, np.ndarray):
            raise ValueError("Input X must be a numpy array")
        if X.ndim != 2:
            raise ValueError("Input X must be 2-dimensional")
        if X.shape[1] != self.layers[0]:
            raise ValueError(f"Input features {X.shape[1]} don't match expected {self.layers[0]}")
        
        # Initialize cache
        self.cache = {'A': [X]}
        current_input = X
        
        # Forward through all layers
        for i in range(self.n_layers - 1):
            # Linear transformation: z = Wa + b
            z = np.dot(current_input, self.weights[i]) + self.biases[i]
            self.cache[f'Z{i+1}'] = z
            
            # Activation function
            if i == self.n_layers - 2:  # Output layer
                a = softmax_stable(z)
            else:  # Hidden layers
                a = activation_function(z, self.activation)
                # Apply dropout only during training and to hidden layers
                a = apply_dropout(a, self.dropout_rate, training)
            
            self.cache['A'].append(a)
            current_input = a
        
        return current_input
    
    def _compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Compute cross-entropy loss with regularization
        
        Parameters:
        -----------
        y_true : np.ndarray
            True labels (one-hot encoded)
        y_pred : np.ndarray
            Predicted probabilities
            
        Returns:
        --------
        float
            Total loss (cross-entropy + regularization)
        """
        m = y_true.shape[0]
        
        # Cross-entropy loss with numerical stability
        epsilon = 1e-15  # Prevent log(0)
        y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
        ce_loss = -np.mean(np.sum(y_true * np.log(y_pred_clipped), axis=1))
        
        # Add regularization
        reg_loss = 0.0
        if self.regularization == 'l1':
            reg_loss = self.reg_lambda * sum(np.sum(np.abs(w)) for w in self.weights)
        elif self.regularization == 'l2':
            reg_loss = self.reg_lambda * sum(np.sum(w ** 2) for w in self.weights)
        
        return ce_loss + reg_loss
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Make predictions on new data
        
        Parameters:
        -----------
        X : np.ndarray
            Input data
            
        Returns:
        --------
        np.ndarray
            Predicted class labels
        """
        y_pred_proba = self.forward(X, training=False)
        return np.argmax(y_pred_proba, axis=1)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Get prediction probabilities
        
        Parameters:
        -----------
        X : np.ndarray
            Input data
            
        Returns:
        --------
        np.ndarray
            Prediction probabilities
        """
        return self.forward(X, training=False)
    
    # Add methods to the AdvancedMLP class
    AdvancedMLP.forward = forward
    AdvancedMLP._compute_loss = _compute_loss
    AdvancedMLP.predict = predict
    AdvancedMLP.predict_proba = predict_proba

# Apply the methods to the class
forward_propagation_methods()

print("🚀 Forward propagation methods added")
print("📊 Loss computation with regularization ready")
print("🎯 Prediction methods implemented")
print("💾 Caching system for efficient backpropagation")

## 4. Forward Propagation Methods

This section implements the forward propagation mechanism with caching for efficient backpropagation.

### Mathematical Foundation:
For each layer $l$:
1. **Linear Transformation**: $\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$
2. **Activation**: $\mathbf{a}^{[l]} = g(\mathbf{z}^{[l]})$ where $g$ is the activation function
3. **Output Layer**: Uses softmax for multi-class classification

### Key Features:
- **Caching**: Stores intermediate values ($\mathbf{z}$, $\mathbf{a}$) for backpropagation
- **Dropout**: Applied during training for regularization
- **Numerically Stable**: Softmax with max subtraction for stability
- **Batch Processing**: Efficient vectorized operations

In [ ]:
class AdvancedMLP:
    """
    Comprehensive Multi-Layer Perceptron implementation with advanced features.
    
    This implementation includes:
    - Multiple optimizers (SGD, Momentum, Adam, RMSprop, AdaGrad)
    - Various activation functions (ReLU, Sigmoid, Tanh, Leaky ReLU)
    - Regularization techniques (L1, L2, Dropout)
    - Advanced initialization methods (Xavier, He)
    - Learning rate scheduling
    - Training diagnostics and visualization
    """
    
    def __init__(self, 
                 layers: List[int], 
                 activation: str = 'relu', 
                 optimizer: str = 'adam', 
                 learning_rate: float = 0.001,
                 regularization: Optional[str] = None, 
                 reg_lambda: float = 0.01, 
                 dropout_rate: float = 0.0,
                 batch_size: int = 32, 
                 random_state: int = 42) -> None:
        """
        Initialize the Advanced MLP with comprehensive configuration options.
        
        Parameters:
        -----------
        layers : List[int]
            List of layer sizes [input_size, hidden1, hidden2, ..., output_size]
        activation : str, default='relu'
            Activation function ('relu', 'sigmoid', 'tanh', 'leaky_relu')
        optimizer : str, default='adam'
            Optimization algorithm ('sgd', 'momentum', 'adam', 'rmsprop', 'adagrad')
        learning_rate : float, default=0.001
            Initial learning rate
        regularization : str or None, default=None
            Regularization type ('l1', 'l2', None)
        reg_lambda : float, default=0.01
            Regularization strength
        dropout_rate : float, default=0.0
            Dropout probability (0.0 = no dropout)
        batch_size : int, default=32
            Mini-batch size for training
        random_state : int, default=42
            Random seed for reproducibility
            
        Raises:
        -------
        ValueError
            If parameters are invalid
        """
        # Validate inputs
        if len(layers) < 2:
            raise ValueError("Must specify at least input and output layer sizes")
        if any(size <= 0 for size in layers):
            raise ValueError("All layer sizes must be positive")
        if activation not in ['relu', 'sigmoid', 'tanh', 'leaky_relu']:
            raise ValueError(f"Unsupported activation: {activation}")
        if optimizer not in ['sgd', 'momentum', 'adam', 'rmsprop', 'adagrad']:
            raise ValueError(f"Unsupported optimizer: {optimizer}")
        if regularization not in [None, 'l1', 'l2']:
            raise ValueError(f"Unsupported regularization: {regularization}")
        if not 0 <= dropout_rate < 1:
            raise ValueError("Dropout rate must be in [0, 1)")
        if learning_rate <= 0:
            raise ValueError("Learning rate must be positive")
        
        np.random.seed(random_state)
        
        # Store configuration
        self.layers = layers
        self.n_layers = len(layers)
        self.activation = activation
        self.optimizer = optimizer
        self.learning_rate = learning_rate
        self.initial_lr = learning_rate
        self.regularization = regularization
        self.reg_lambda = reg_lambda
        self.dropout_rate = dropout_rate
        self.batch_size = batch_size
        self.random_state = random_state
        
        # Initialize parameters
        self._initialize_parameters()
        self._initialize_optimizer()
        
        # Training history
        self.history: Dict[str, List[float]] = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'lr_history': [], 'gradient_norms': []
        }
        
        # Cache for forward pass
        self.cache: Dict[str, Any] = {}
        
    def _initialize_parameters(self) -> None:
        """Initialize weights and biases using Xavier/He initialization"""
        self.weights: List[np.ndarray] = []
        self.biases: List[np.ndarray] = []
        
        for i in range(self.n_layers - 1):
            # Choose initialization strategy based on activation
            if self.activation in ['relu', 'leaky_relu']:
                # He initialization for ReLU-like activations
                std = np.sqrt(2.0 / self.layers[i])
            else:
                # Xavier initialization for sigmoid/tanh activations
                std = np.sqrt(2.0 / (self.layers[i] + self.layers[i + 1]))
            
            W = np.random.normal(0, std, (self.layers[i], self.layers[i + 1]))
            b = np.zeros((1, self.layers[i + 1]))
            
            self.weights.append(W)
            self.biases.append(b)
    
    def _initialize_optimizer(self) -> None:
        """Initialize optimizer-specific parameters"""
        if self.optimizer in ['momentum', 'adam']:
            # Momentum terms for weights and biases
            self.momentum_w: List[np.ndarray] = [np.zeros_like(w) for w in self.weights]
            self.momentum_b: List[np.ndarray] = [np.zeros_like(b) for b in self.biases]
            
        if self.optimizer in ['adam', 'rmsprop', 'adagrad']:
            # Second moment terms for weights and biases
            self.v_w: List[np.ndarray] = [np.zeros_like(w) for w in self.weights]
            self.v_b: List[np.ndarray] = [np.zeros_like(b) for b in self.biases]
            
        # Optimizer hyperparameters
        self.beta1: float = 0.9   # For Adam
        self.beta2: float = 0.999 # For Adam
        self.epsilon: float = 1e-8
        self.t: int = 0  # Time step for Adam bias correction
    
    def get_parameters(self) -> Dict[str, List[np.ndarray]]:
        """
        Return current parameters
        
        Returns:
        --------
        Dict[str, List[np.ndarray]]
            Dictionary containing weights and biases
        """
        return {'weights': self.weights, 'biases': self.biases}
    
    def get_config(self) -> Dict[str, Any]:
        """
        Get model configuration
        
        Returns:
        --------
        Dict[str, Any]
            Model configuration dictionary
        """
        return {
            'layers': self.layers,
            'activation': self.activation,
            'optimizer': self.optimizer,
            'learning_rate': self.initial_lr,
            'regularization': self.regularization,
            'reg_lambda': self.reg_lambda,
            'dropout_rate': self.dropout_rate,
            'batch_size': self.batch_size,
            'random_state': self.random_state
        }

print("🏗️  AdvancedMLP class structure initialized")
print("⚙️  Parameter initialization methods ready")
print("🔧 Optimizer setup complete")
print("📊 Training history tracking prepared")

## 3. AdvancedMLP Class Initialization and Core Methods

This section defines the main AdvancedMLP class with comprehensive initialization and core structural methods.

### Key Features:
- **Flexible Architecture**: Support for arbitrary network depths and widths
- **Parameter Initialization**: Xavier/Glorot and He initialization strategies
- **Optimizer Setup**: Initialize optimizer-specific parameters
- **Training History**: Track metrics throughout training
- **Comprehensive Configuration**: All hyperparameters with sensible defaults

### Initialization Strategies:
- **Xavier/Glorot**: $\text{std} = \sqrt{\frac{2}{n_{in} + n_{out}}}$ - Good for sigmoid/tanh
- **He**: $\text{std} = \sqrt{\frac{2}{n_{in}}}$ - Optimal for ReLU networks

In [ ]:
# Activation functions and utility functions
def activation_function(z: np.ndarray, activation: str, derivative: bool = False) -> np.ndarray:
    """
    Apply activation function with derivative option
    
    Parameters:
    -----------
    z : np.ndarray
        Input array
    activation : str
        Type of activation function ('relu', 'leaky_relu', 'sigmoid', 'tanh')
    derivative : bool
        Whether to compute derivative
        
    Returns:
    --------
    np.ndarray
        Activated values or derivatives
    """
    if activation == 'relu':
        if derivative:
            return (z > 0).astype(float)
        return np.maximum(0, z)
        
    elif activation == 'leaky_relu':
        alpha = 0.01
        if derivative:
            return np.where(z > 0, 1, alpha)
        return np.where(z > 0, z, alpha * z)
        
    elif activation == 'sigmoid':
        if derivative:
            s = expit(z)
            return s * (1 - s)
        return expit(z)
        
    elif activation == 'tanh':
        if derivative:
            return 1 - np.tanh(z) ** 2
        return np.tanh(z)
    
    else:
        raise ValueError(f"Unsupported activation function: {activation}")

def softmax_stable(z: np.ndarray) -> np.ndarray:
    """
    Numerically stable softmax activation
    
    Parameters:
    -----------
    z : np.ndarray
        Input logits
        
    Returns:
    --------
    np.ndarray
        Softmax probabilities
    """
    # Subtract max for numerical stability
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def apply_dropout(a: np.ndarray, dropout_rate: float, training: bool = True) -> np.ndarray:
    """
    Apply dropout regularization
    
    Parameters:
    -----------
    a : np.ndarray
        Input activations
    dropout_rate : float
        Probability of dropping neurons
    training : bool
        Whether in training mode
        
    Returns:
    --------
    np.ndarray
        Activations with dropout applied
    """
    if training and dropout_rate > 0:
        dropout_mask = np.random.binomial(1, 1 - dropout_rate, size=a.shape)
        return a * dropout_mask / (1 - dropout_rate)
    return a

def validate_input_data(X: np.ndarray, y: np.ndarray) -> None:
    """
    Validate input data for training
    
    Parameters:
    -----------
    X : np.ndarray
        Input features
    y : np.ndarray
        Target labels
        
    Raises:
    -------
    ValueError
        If data is invalid
    """
    if not isinstance(X, np.ndarray) or not isinstance(y, np.ndarray):
        raise ValueError("X and y must be numpy arrays")
    
    if X.ndim != 2:
        raise ValueError("X must be a 2D array")
    
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of samples")
    
    if np.any(np.isnan(X)) or np.any(np.isnan(y)):
        raise ValueError("Data contains NaN values")
    
    if np.any(np.isinf(X)) or np.any(np.isinf(y)):
        raise ValueError("Data contains infinite values")

print("✅ Activation functions and utilities defined")
print("🔧 Input validation functions ready")
print("📐 Numerical stability ensured")

## 2. Activation Functions and Utilities

This section defines various activation functions and their derivatives, plus utility functions used throughout the implementation.

### Activation Functions Available:
- **ReLU**: $f(x) = \max(0, x)$ - Most commonly used, good gradient flow
- **Leaky ReLU**: $f(x) = \max(\alpha x, x)$ - Prevents dead neurons
- **Sigmoid**: $f(x) = \frac{1}{1 + e^{-x}}$ - Smooth, outputs (0,1)
- **Tanh**: $f(x) = \tanh(x)$ - Smooth, outputs (-1,1)

### Key Features:
- Numerically stable implementations
- Efficient derivative computation
- Vectorized operations for batch processing

In [ ]:
# Mathematical foundations and comprehensive imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
from scipy.special import expit, softmax
import warnings
from typing import List, Tuple, Optional, Union, Dict, Any

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

print("🧮 Mathematical Foundations Loaded")
print("📚 Libraries imported successfully")
print("=" * 50)

## 1. Mathematical Foundations and Imports

This section contains all the necessary imports and establishes the mathematical foundation for our advanced MLP implementation.

### Key Mathematical Components:
- **Forward Propagation**: $\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$, $\mathbf{a}^{[l]} = g(\mathbf{z}^{[l]})$
- **Loss Function**: Cross-entropy loss with regularization
- **Backpropagation**: Chain rule for gradient computation
- **Optimization**: Various algorithms for parameter updates

# Advanced MLP Implementation - Production Ready

Now that we've seen the basic concepts, let's explore a comprehensive, production-ready Multi-Layer Perceptron implementation with advanced features:

## 🚀 Advanced Features:
- **Multiple Optimizers**: SGD, Momentum, Adam, RMSprop, AdaGrad
- **Various Activation Functions**: ReLU, Sigmoid, Tanh, Leaky ReLU
- **Regularization Techniques**: L1, L2, Dropout
- **Advanced Initialization**: Xavier/Glorot, He initialization
- **Learning Rate Scheduling**: Step decay, exponential decay, cosine annealing
- **Training Diagnostics**: Loss tracking, gradient monitoring, early stopping
- **Comprehensive Validation**: Input validation, type hints, error handling

## 📋 Organization:
The implementation is organized into logical sections:
1. **Mathematical Foundations and Imports**
2. **Activation Functions and Utilities**
3. **AdvancedMLP Class Initialization and Core Methods**
4. **Forward Propagation Methods**
5. **Backpropagation and Optimization Methods**
6. **Training and Evaluation Methods**
7. **Visualization Functions**
8. **Example Demonstrations**

Let's dive into each section!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class SimpleMLP:
    """
    A basic Multi-Layer Perceptron for binary classification
    
    This implementation demonstrates core concepts:
    - Forward propagation through layers
    - Sigmoid activation and cross-entropy loss
    - Backpropagation with gradient computation
    - Basic SGD optimization
    """
    
    def __init__(self, input_size: int, hidden_size: int, output_size: int, learning_rate: float = 0.01):
        """
        Initialize the simple MLP
        
        Parameters:
        -----------
        input_size : int
            Number of input features
        hidden_size : int  
            Number of neurons in hidden layer
        output_size : int
            Number of output classes
        learning_rate : float
            Learning rate for SGD optimization
        """
        # Initialize weights with small random values
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        
        self.learning_rate = learning_rate
        
    def sigmoid(self, z: np.ndarray) -> np.ndarray:
        """Sigmoid activation function with numerical stability"""
        z = np.clip(z, -500, 500)  # Prevent overflow
        return 1 / (1 + np.exp(-z))
    
    def sigmoid_derivative(self, z: np.ndarray) -> np.ndarray:
        """Derivative of sigmoid function"""
        s = self.sigmoid(z)
        return s * (1 - s)
    
    def forward(self, X: np.ndarray) -> tuple:
        """
        Forward propagation
        
        Parameters:
        -----------
        X : np.ndarray
            Input data of shape (batch_size, input_size)
            
        Returns:
        --------
        tuple
            (z1, a1, z2, a2) - intermediate values and final output
        """
        # Hidden layer
        z1 = np.dot(X, self.W1) + self.b1
        a1 = self.sigmoid(z1)
        
        # Output layer  
        z2 = np.dot(a1, self.W2) + self.b2
        a2 = self.sigmoid(z2)
        
        return z1, a1, z2, a2
    
    def compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Compute binary cross-entropy loss
        
        Parameters:
        -----------
        y_true : np.ndarray
            True labels
        y_pred : np.ndarray
            Predicted probabilities
            
        Returns:
        --------
        float
            Average loss across the batch
        """
        m = y_true.shape[0]
        
        # Clip predictions to prevent log(0)
        y_pred_clipped = np.clip(y_pred, 1e-15, 1 - 1e-15)
        
        # Binary cross-entropy
        loss = -np.mean(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))
        
        return loss
    
    def backward(self, X: np.ndarray, y: np.ndarray, z1: np.ndarray, a1: np.ndarray, 
                 z2: np.ndarray, a2: np.ndarray) -> None:
        """
        Backward propagation and parameter updates
        
        Parameters:
        -----------
        X : np.ndarray
            Input data
        y : np.ndarray
            True labels
        z1, a1, z2, a2 : np.ndarray
            Values from forward pass
        """
        m = X.shape[0]
        
        # Output layer gradients
        dz2 = a2 - y
        dW2 = np.dot(a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m
        
        # Hidden layer gradients
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * self.sigmoid_derivative(z1)
        dW1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m
        
        # Update parameters
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
    
    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 1000, verbose: bool = True) -> list:
        """
        Train the MLP
        
        Parameters:
        -----------
        X : np.ndarray
            Training data
        y : np.ndarray
            Training labels
        epochs : int
            Number of training epochs
        verbose : bool
            Whether to print progress
            
        Returns:
        --------
        list
            Training loss history
        """
        losses = []
        
        for epoch in range(epochs):
            # Forward pass
            z1, a1, z2, a2 = self.forward(X)
            
            # Compute loss
            loss = self.compute_loss(y, a2)
            losses.append(loss)
            
            # Backward pass
            self.backward(X, y, z1, a1, z2, a2)
            
            # Print progress
            if verbose and epoch % 100 == 0:
                accuracy = self.score(X, y)
                print(f"Epoch {epoch:4d}: Loss = {loss:.6f}, Accuracy = {accuracy:.4f}")
        
        return losses
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions"""
        _, _, _, a2 = self.forward(X)
        return (a2 > 0.5).astype(int)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Get prediction probabilities"""
        _, _, _, a2 = self.forward(X)
        return a2
    
    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute accuracy"""
        predictions = self.predict(X)
        return np.mean(predictions.ravel() == y.ravel())

# Example usage
print("🧠 Simple MLP Implementation")
print("=" * 40)

# Generate sample data
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                          n_redundant=5, n_classes=2, random_state=42)

# Preprocess data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Initialize and train simple MLP
simple_mlp = SimpleMLP(input_size=20, hidden_size=10, output_size=1, learning_rate=0.1)

print("\n📊 Training Simple MLP...")
losses = simple_mlp.fit(X_train, y_train.reshape(-1, 1), epochs=500, verbose=True)

# Evaluate
train_accuracy = simple_mlp.score(X_train, y_train.reshape(-1, 1))
test_accuracy = simple_mlp.score(X_test, y_test.reshape(-1, 1))

print(f"\n✅ Final Results:")
print(f"   • Training Accuracy: {train_accuracy:.4f}")
print(f"   • Test Accuracy: {test_accuracy:.4f}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', alpha=0.7)
plt.title('Test Data Distribution')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar()

plt.tight_layout()
plt.show()

print("\n🎯 This simple example demonstrates the core MLP concepts!")
print("Now let's explore the advanced implementation with more features...")

# Simple MLP Example - Basic Implementation

Before diving into the advanced implementation, let's start with a simple, basic Multi-Layer Perceptron to understand the core concepts:

## Key Concepts:
- **Forward Propagation**: Data flows from input to output through layers
- **Activation Functions**: Non-linear transformations (ReLU, Sigmoid, etc.)  
- **Backpropagation**: Computing gradients using the chain rule
- **Optimization**: Updating weights to minimize loss

This simple example will help you understand the fundamental building blocks before exploring the comprehensive implementation.

<div align="center">

# 🧠 Multi-Layer Perceptron (MLP)
## Comprehensive Guide & Advanced Implementation

[\![Neural Network](https://img.shields.io/badge/Neural_Network-MLP-blue.svg?style=for-the-badge&logo=tensorflow)](https://github.com)
[\![Python](https://img.shields.io/badge/Python-3.8+-green.svg?style=for-the-badge&logo=python)](https://python.org)
[\![Mathematics](https://img.shields.io/badge/Mathematics-Advanced-red.svg?style=for-the-badge&logo=mathworks)](https://mathworks.com)

\![Neural Network Animation](https://miro.medium.com/v2/1*pO5X2c28F1ysJhwnmPsy3Q.gif)

</div>

---

## 🎯 Learning Objectives

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 15px; color: white; margin: 15px 0;">

### By the end of this notebook, you will master:

<table style="width: 100%; color: white;">
<tr>
<td style="width: 50%; vertical-align: top;">

**🧮 Mathematical Foundations**
- Complete mathematical theory of MLPs
- Universal approximation theorem
- Forward & backward propagation derivations

**⚙️ Optimization Techniques** 
- SGD, Adam, RMSprop, AdaGrad
- Learning rate scheduling strategies
- Advanced gradient descent variants

**🛡️ Regularization Methods**
- Dropout, Batch Normalization
- Weight Decay (L1/L2)
- Early stopping techniques

</td>
<td style="width: 50%; vertical-align: top;">

**🎛️ Activation Functions**
- ReLU, Leaky ReLU, Sigmoid, Tanh
- Function analysis & comparison
- Impact on learning dynamics

**🏗️ Architecture Design**
- Design principles for different problems
- Hyperparameter tuning strategies
- Training dynamics & diagnostics

**🚀 Real-world Applications**
- Comprehensive benchmarking
- Advanced topics & best practices
- Production-ready implementations

</td>
</tr>
</table>

</div>

# Backpropagation Algorithm: Complete Mathematical Derivation

## 5. The Backpropagation Algorithm

Backpropagation is the **cornerstone algorithm** for training neural networks. It efficiently computes gradients of the loss function with respect to all parameters using the **chain rule of calculus**.

### 5.1 Mathematical Foundation: The Chain Rule

For a composite function $f(g(h(x)))$, the chain rule states:
$$\\frac{df}{dx} = \\frac{df}{dg} \\cdot \\frac{dg}{dh} \\cdot \\frac{dh}{dx}$$

In neural networks, this becomes:
$$\\frac{\\partial J}{\\partial \\mathbf{W}^{[l]}} = \\frac{\\partial J}{\\partial \\mathbf{z}^{[l]}} \\cdot \\frac{\\partial \\mathbf{z}^{[l]}}{\\partial \\mathbf{W}^{[l]}}$$

### 5.2 Backpropagation Derivation: Step by Step

#### **Step 1: Output Layer Gradients (Layer L)**

Starting from the loss function $J$ and working backwards:

**For Cross-Entropy Loss with Softmax**:
$$\\frac{\\partial J}{\\partial \\mathbf{z}^{[L]}} = \\mathbf{a}^{[L]} - \\mathbf{y}$$

This remarkably simple result comes from the mathematical properties of softmax and cross-entropy loss.

**Weight Gradients**:
$$\\frac{\\partial J}{\\partial \\mathbf{W}^{[L]}} = \\frac{\\partial J}{\\partial \\mathbf{z}^{[L]}} \\cdot (\\mathbf{a}^{[L-1]})^T = (\\mathbf{a}^{[L]} - \\mathbf{y}) \\cdot (\\mathbf{a}^{[L-1]})^T$$

**Bias Gradients**:
$$\\frac{\\partial J}{\\partial \\mathbf{b}^{[L]}} = \\frac{\\partial J}{\\partial \\mathbf{z}^{[L]}} = \\mathbf{a}^{[L]} - \\mathbf{y}$$

#### **Step 2: Hidden Layer Gradients (Layers l < L)**

**Error Propagation**:
$$\\frac{\\partial J}{\\partial \\mathbf{z}^{[l]}} = (\\mathbf{W}^{[l+1]})^T \\cdot \\frac{\\partial J}{\\partial \\mathbf{z}^{[l+1]}} \\odot g'(\\mathbf{z}^{[l]})$$

Where $\\odot$ denotes element-wise multiplication and $g'(\\mathbf{z}^{[l]})$ is the derivative of the activation function.

**Weight Gradients**:
$$\\frac{\\partial J}{\\partial \\mathbf{W}^{[l]}} = \\frac{\\partial J}{\\partial \\mathbf{z}^{[l]}} \\cdot (\\mathbf{a}^{[l-1]})^T$$

**Bias Gradients**:
$$\\frac{\\partial J}{\\partial \\mathbf{b}^{[l]}} = \\frac{\\partial J}{\\partial \\mathbf{z}^{[l]}}$$

### 5.3 Activation Function Derivatives

#### **ReLU Derivative**
$$\\frac{\\partial}{\\partial z} \\text{ReLU}(z) = \\begin{cases} 
1 & \\text{if } z > 0 \\\\
0 & \\text{if } z \\leq 0
\\end{cases}$$

#### **Sigmoid Derivative**
$$\\frac{\\partial}{\\partial z} \\sigma(z) = \\sigma(z)(1 - \\sigma(z))$$

#### **Tanh Derivative**
$$\\frac{\\partial}{\\partial z} \\tanh(z) = 1 - \\tanh^2(z)$$

### 5.4 Vectorized Backpropagation Algorithm

**Input**: Training batch $\\mathbf{X} \\in \\mathbb{R}^{n_0 \\times m}$, labels $\\mathbf{Y} \\in \\mathbb{R}^{n_L \\times m}$

**Forward Pass**: Compute $\\mathbf{A}^{[l]}$ and $\\mathbf{Z}^{[l]}$ for $l = 1, \\ldots, L$

**Backward Pass**:
1. **Initialize output error**:
   $$\\mathbf{dZ}^{[L]} = \\mathbf{A}^{[L]} - \\mathbf{Y}$$

2. **For layer** $l = L, L-1, \\ldots, 1$:
   $$\\mathbf{dW}^{[l]} = \\frac{1}{m} \\mathbf{dZ}^{[l]} \\cdot (\\mathbf{A}^{[l-1]})^T$$
   $$\\mathbf{db}^{[l]} = \\frac{1}{m} \\text{sum}(\\mathbf{dZ}^{[l]}, \\text{axis}=1, \\text{keepdims}=\\text{True})$$
   
   If $l > 1$:
   $$\\mathbf{dA}^{[l-1]} = (\\mathbf{W}^{[l]})^T \\cdot \\mathbf{dZ}^{[l]}$$
   $$\\mathbf{dZ}^{[l-1]} = \\mathbf{dA}^{[l-1]} \\odot g'^{[l-1]}(\\mathbf{Z}^{[l-1]})$$

### 5.5 Computational Complexity

**Forward Pass**: $O(\\sum_{l=1}^{L} n_{l-1} \\cdot n_l)$
**Backward Pass**: $O(\\sum_{l=1}^{L} n_{l-1} \\cdot n_l)$

**Key Insight**: Backpropagation has the **same computational complexity** as forward propagation!

---

## 6. Advanced Optimization Algorithms

### 6.1 Gradient Descent Variants

#### **Batch Gradient Descent**
$$\\mathbf{W}^{[l]} := \\mathbf{W}^{[l]} - \\alpha \\frac{\\partial J}{\\partial \\mathbf{W}^{[l]}}$$

**Properties**:
- ✅ Guaranteed convergence to global minimum (convex case)
- ✅ Stable parameter updates
- ⚠️ Slow for large datasets
- ⚠️ Can get stuck in local minima (non-convex case)

#### **Stochastic Gradient Descent (SGD)**
Process one example at a time:
$$\\mathbf{W}^{[l]} := \\mathbf{W}^{[l]} - \\alpha \\frac{\\partial \\mathcal{L}^{(i)}}{\\partial \\mathbf{W}^{[l]}}$$

**Properties**:
- ✅ Fast updates, good for large datasets
- ✅ Can escape local minima due to noise
- ⚠️ High variance in updates
- ⚠️ May not converge exactly

#### **Mini-batch Gradient Descent**
Process mini-batches of size $B$:
$$\\mathbf{W}^{[l]} := \\mathbf{W}^{[l]} - \\alpha \\frac{1}{B} \\sum_{i \\in \\text{batch}} \\frac{\\partial \\mathcal{L}^{(i)}}{\\partial \\mathbf{W}^{[l]}}$$

**Properties**:
- ✅ Balance between stability and efficiency
- ✅ Vectorization benefits
- ✅ Good convergence properties

### 6.2 Momentum-Based Methods

#### **SGD with Momentum**
$$\\mathbf{v}_t = \\beta \\mathbf{v}_{t-1} + (1-\\beta) \\nabla J(\\mathbf{W}_t)$$
$$\\mathbf{W}_{t+1} = \\mathbf{W}_t - \\alpha \\mathbf{v}_t$$

**Intuition**: Add "momentum" to parameter updates, helping to:
- Accelerate convergence in consistent gradient directions
- Dampen oscillations in high-curvature directions

#### **Nesterov Accelerated Gradient (NAG)**
$$\\mathbf{v}_t = \\beta \\mathbf{v}_{t-1} + \\nabla J(\\mathbf{W}_t - \\alpha \\beta \\mathbf{v}_{t-1})$$
$$\\mathbf{W}_{t+1} = \\mathbf{W}_t - \\alpha \\mathbf{v}_t$$

**Key Difference**: Compute gradient at the "lookahead" position.

### 6.3 Adaptive Learning Rate Methods

#### **AdaGrad (Adaptive Gradient)**
$$\\mathbf{G}_t = \\mathbf{G}_{t-1} + (\\nabla J(\\mathbf{W}_t))^2$$
$$\\mathbf{W}_{t+1} = \\mathbf{W}_t - \\frac{\\alpha}{\\sqrt{\\mathbf{G}_t + \\epsilon}} \\odot \\nabla J(\\mathbf{W}_t)$$

**Properties**:
- ✅ Adapts learning rate per parameter
- ✅ Good for sparse gradients
- ⚠️ Learning rate decays too aggressively

#### **RMSprop**
$$\\mathbf{v}_t = \\beta \\mathbf{v}_{t-1} + (1-\\beta)(\\nabla J(\\mathbf{W}_t))^2$$
$$\\mathbf{W}_{t+1} = \\mathbf{W}_t - \\frac{\\alpha}{\\sqrt{\\mathbf{v}_t + \\epsilon}} \\odot \\nabla J(\\mathbf{W}_t)$$

**Improvement**: Uses exponential moving average instead of cumulative sum.

#### **Adam (Adaptive Moment Estimation)**
$$\\mathbf{m}_t = \\beta_1 \\mathbf{m}_{t-1} + (1-\\beta_1) \\nabla J(\\mathbf{W}_t)$$
$$\\mathbf{v}_t = \\beta_2 \\mathbf{v}_{t-1} + (1-\\beta_2) (\\nabla J(\\mathbf{W}_t))^2$$

**Bias Correction**:
$$\\hat{\\mathbf{m}}_t = \\frac{\\mathbf{m}_t}{1-\\beta_1^t}, \\quad \\hat{\\mathbf{v}}_t = \\frac{\\mathbf{v}_t}{1-\\beta_2^t}$$

**Parameter Update**:
$$\\mathbf{W}_{t+1} = \\mathbf{W}_t - \\frac{\\alpha}{\\sqrt{\\hat{\\mathbf{v}}_t} + \\epsilon} \\hat{\\mathbf{m}}_t$$

**Default Hyperparameters**: $\\beta_1 = 0.9$, $\\beta_2 = 0.999$, $\\epsilon = 10^{-8}$

### 6.4 Learning Rate Scheduling

#### **Step Decay**
$$\\alpha_t = \\alpha_0 \\cdot \\gamma^{\\lfloor t/s \\rfloor}$$

#### **Exponential Decay**
$$\\alpha_t = \\alpha_0 \\cdot e^{-kt}$$

#### **Cosine Annealing**
$$\\alpha_t = \\alpha_{min} + \\frac{1}{2}(\\alpha_{max} - \\alpha_{min})(1 + \\cos(\\frac{t\\pi}{T}))$$

### 6.5 Parameter Initialization Strategies

#### **Xavier/Glorot Initialization**
For layers with $n_{in}$ inputs and $n_{out}$ outputs:
$$\\mathbf{W} \\sim \\mathcal{U}\\left(-\\sqrt{\\frac{6}{n_{in} + n_{out}}}, \\sqrt{\\frac{6}{n_{in} + n_{out}}}\\right)$$

**Motivation**: Keep variance of activations and gradients roughly equal across layers.

#### **He Initialization**
$$\\mathbf{W} \\sim \\mathcal{N}\\left(0, \\frac{2}{n_{in}}\\right)$$

**Better for ReLU networks**: Accounts for the fact that ReLU kills negative activations.

---

## 7. Training Dynamics and Common Issues

### 7.1 Vanishing Gradient Problem

**Problem**: Gradients become exponentially small as they propagate backwards through deep networks.

**Mathematical Analysis**:
$$\\frac{\\partial J}{\\partial \\mathbf{W}^{[1]}} = \\frac{\\partial J}{\\partial \\mathbf{z}^{[L]}} \\prod_{l=2}^{L} \\mathbf{W}^{[l]} \\odot g'(\\mathbf{z}^{[l-1]})$$

If $|\\mathbf{W}^{[l]}| < 1$ and $|g'(\\mathbf{z}^{[l]})| < 1$, then gradients vanish exponentially.

**Solutions**:
- Use ReLU activations (constant gradient for $z > 0$)
- Proper weight initialization (Xavier/He)
- Batch normalization
- Residual connections (skip connections)

### 7.2 Exploding Gradient Problem

**Problem**: Gradients become exponentially large, causing unstable training.

**Solutions**:
- Gradient clipping: $\\mathbf{g} = \\min(1, \\frac{\\tau}{\\|\\mathbf{g}\\|}) \\mathbf{g}$
- Proper weight initialization
- Lower learning rates

### 7.3 Dead ReLU Problem

**Problem**: ReLU neurons can "die" and stop learning if they consistently output zero.

**Causes**:
- Large negative biases
- Too high learning rates
- Poor initialization

**Solutions**:
- Use Leaky ReLU or ELU
- Proper initialization
- Lower learning rates
- Batch normalization

---

This comprehensive mathematical foundation provides the theoretical understanding necessary to implement and debug MLPs effectively. The next sections will cover practical implementation and advanced techniques.

# Advanced MLP Implementation with Multiple Optimizers

## 8. Complete MLP Implementation from Scratch

This section provides a comprehensive, production-ready implementation of Multi-Layer Perceptron with:
- **Multiple optimization algorithms** (SGD, Adam, RMSprop, AdaGrad)
- **Various activation functions** (ReLU, Sigmoid, Tanh, Leaky ReLU)
- **Regularization techniques** (L1, L2, Dropout)
- **Advanced initialization methods** (Xavier, He)
- **Learning rate scheduling**
- **Training diagnostics and visualization**

### 8.1 Mathematical Foundations Recap

**Forward Propagation**:
- Linear: $\\mathbf{z}^{[l]} = \\mathbf{W}^{[l]} \\mathbf{a}^{[l-1]} + \\mathbf{b}^{[l]}$
- Activation: $\\mathbf{a}^{[l]} = g(\\mathbf{z}^{[l]})$

**Backward Propagation**:
- Output layer: $\\frac{\\partial J}{\\partial \\mathbf{z}^{[L]}} = \\mathbf{a}^{[L]} - \\mathbf{y}$
- Hidden layers: $\\frac{\\partial J}{\\partial \\mathbf{z}^{[l]}} = (\\mathbf{W}^{[l+1]})^T \\frac{\\partial J}{\\partial \\mathbf{z}^{[l+1]}} \\odot g'(\\mathbf{z}^{[l]})$

**Optimization Update**:
- SGD: $\\mathbf{W} := \\mathbf{W} - \\alpha \\nabla \\mathbf{W}$
- Adam: $\\mathbf{W} := \\mathbf{W} - \\frac{\\alpha}{\\sqrt{\\hat{\\mathbf{v}}} + \\epsilon} \\hat{\\mathbf{m}}$

### 8.2 Key Implementation Features

1. **Vectorized Operations**: Efficient NumPy implementation for batch processing
2. **Flexible Architecture**: Support for arbitrary network depths and widths  
3. **Multiple Optimizers**: SGD, Momentum, Adam, RMSprop, AdaGrad
4. **Regularization**: L1/L2 weight decay, Dropout during training
5. **Initialization**: Xavier (Glorot) and He initialization strategies
6. **Training Monitoring**: Loss tracking, accuracy computation, gradient monitoring
7. **Early Stopping**: Prevent overfitting with validation-based stopping
8. **Learning Rate Scheduling**: Step decay, exponential decay, cosine annealing

### 8.3 Performance Optimizations

- **Mini-batch Processing**: Vectorized computations across samples
- **Activation Caching**: Store forward pass results for efficient backprop
- **Gradient Clipping**: Prevent exploding gradients
- **Numerical Stability**: Careful handling of log/exp operations
- **Memory Efficiency**: Minimal storage of intermediate results

### 8.4 Usage Philosophy

The implementation follows these principles:
- **Modularity**: Clear separation of concerns (forward, backward, optimization)
- **Extensibility**: Easy to add new activations, optimizers, or regularization
- **Diagnostics**: Comprehensive training insights and debugging tools
- **Standards Compliance**: Follows deep learning best practices and conventions

This implementation serves as both a learning tool and a practical neural network library suitable for research and experimentation on medium-scale problems.

In [ ]:
# 🧠 ADVANCED MLP IMPLEMENTATION WITH COMPREHENSIVE FEATURES
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
from scipy.special import expit, softmax
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

print("🎯 Advanced Multi-Layer Perceptron Implementation")
print("=" * 55)

class AdvancedMLP:
    """
    Comprehensive Multi-Layer Perceptron implementation with:
    - Multiple optimizers (SGD, Momentum, Adam, RMSprop, AdaGrad)
    - Various activation functions (ReLU, Sigmoid, Tanh, Leaky ReLU)
    - Regularization techniques (L1, L2, Dropout)
    - Advanced initialization methods (Xavier, He)
    - Learning rate scheduling
    - Training diagnostics and visualization
    """
    
    def __init__(self, layers, activation='relu', optimizer='adam', learning_rate=0.001,
                 regularization=None, reg_lambda=0.01, dropout_rate=0.0,
                 batch_size=32, random_state=42):
        """
        Initialize the MLP with comprehensive configuration options.
        
        Parameters:
        -----------
        layers : list
            List of layer sizes [input_size, hidden1, hidden2, ..., output_size]
        activation : str
            Activation function ('relu', 'sigmoid', 'tanh', 'leaky_relu')
        optimizer : str
            Optimization algorithm ('sgd', 'momentum', 'adam', 'rmsprop', 'adagrad')
        learning_rate : float
            Initial learning rate
        regularization : str or None
            Regularization type ('l1', 'l2', None)
        reg_lambda : float
            Regularization strength
        dropout_rate : float
            Dropout probability (0.0 = no dropout)
        batch_size : int
            Mini-batch size for training
        random_state : int
            Random seed for reproducibility
        """
        np.random.seed(random_state)
        
        self.layers = layers
        self.n_layers = len(layers)
        self.activation = activation
        self.optimizer = optimizer
        self.learning_rate = learning_rate
        self.initial_lr = learning_rate
        self.regularization = regularization
        self.reg_lambda = reg_lambda
        self.dropout_rate = dropout_rate
        self.batch_size = batch_size
        self.random_state = random_state
        
        # Initialize parameters
        self._initialize_parameters()
        self._initialize_optimizer()
        
        # Training history
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'lr_history': [], 'gradient_norms': []
        }
        
        # Cache for forward pass
        self.cache = {}
        
    def _initialize_parameters(self):
        """Initialize weights and biases using Xavier/He initialization"""
        self.weights = []
        self.biases = []
        
        for i in range(self.n_layers - 1):
            # Choose initialization strategy based on activation
            if self.activation in ['relu', 'leaky_relu']:
                # He initialization for ReLU-like activations
                std = np.sqrt(2.0 / self.layers[i])
            else:
                # Xavier initialization for sigmoid/tanh activations
                std = np.sqrt(2.0 / (self.layers[i] + self.layers[i + 1]))
            
            W = np.random.normal(0, std, (self.layers[i], self.layers[i + 1]))
            b = np.zeros((1, self.layers[i + 1]))
            
            self.weights.append(W)
            self.biases.append(b)
    
    def _initialize_optimizer(self):
        """Initialize optimizer-specific parameters"""
        if self.optimizer in ['momentum', 'adam']:
            # Momentum terms for weights and biases
            self.momentum_w = [np.zeros_like(w) for w in self.weights]
            self.momentum_b = [np.zeros_like(b) for b in self.biases]
            
        if self.optimizer in ['adam', 'rmsprop', 'adagrad']:
            # Second moment terms for weights and biases
            self.v_w = [np.zeros_like(w) for w in self.weights]
            self.v_b = [np.zeros_like(b) for b in self.biases]
            
        # Optimizer hyperparameters
        self.beta1 = 0.9  # For Adam
        self.beta2 = 0.999  # For Adam
        self.epsilon = 1e-8
        self.t = 0  # Time step for Adam bias correction
    
    def _activation_function(self, z, derivative=False):
        """Apply activation function with derivative option"""
        if self.activation == 'relu':
            if derivative:
                return (z > 0).astype(float)
            return np.maximum(0, z)
            
        elif self.activation == 'leaky_relu':
            alpha = 0.01
            if derivative:
                return np.where(z > 0, 1, alpha)
            return np.where(z > 0, z, alpha * z)
            
        elif self.activation == 'sigmoid':
            if derivative:
                s = expit(z)
                return s * (1 - s)
            return expit(z)
            
        elif self.activation == 'tanh':
            if derivative:
                return 1 - np.tanh(z) ** 2
            return np.tanh(z)
    
    def _softmax(self, z):
        """Numerically stable softmax activation"""
        # Subtract max for numerical stability
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def _apply_dropout(self, a, training=True):
        """Apply dropout regularization"""
        if training and self.dropout_rate > 0:
            dropout_mask = np.random.binomial(1, 1 - self.dropout_rate, size=a.shape)
            return a * dropout_mask / (1 - self.dropout_rate)
        return a
    
    def forward(self, X, training=True):
        """Forward propagation with caching"""
        self.cache = {'A': [X]}
        current_input = X
        
        for i in range(self.n_layers - 1):
            # Linear transformation
            z = np.dot(current_input, self.weights[i]) + self.biases[i]
            self.cache[f'Z{i+1}'] = z
            
            # Activation function
            if i == self.n_layers - 2:  # Output layer
                a = self._softmax(z)
            else:  # Hidden layers
                a = self._activation_function(z)
                a = self._apply_dropout(a, training)
            
            self.cache['A'].append(a)
            current_input = a
        
        return current_input
    
    def _compute_loss(self, y_true, y_pred):
        """Compute cross-entropy loss with regularization"""
        m = y_true.shape[0]
        
        # Cross-entropy loss
        epsilon = 1e-15  # Prevent log(0)
        y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
        ce_loss = -np.mean(np.sum(y_true * np.log(y_pred_clipped), axis=1))
        
        # Regularization
        reg_loss = 0
        if self.regularization == 'l1':
            reg_loss = self.reg_lambda * sum(np.sum(np.abs(w)) for w in self.weights)
        elif self.regularization == 'l2':
            reg_loss = self.reg_lambda * sum(np.sum(w ** 2) for w in self.weights)
        
        return ce_loss + reg_loss
    
    def backward(self, X, y_true, y_pred):
        """Backward propagation with gradient computation"""
        m = X.shape[0]
        gradients_w = []
        gradients_b = []
        
        # Output layer gradient
        dz = y_pred - y_true
        
        # Backward through layers
        for i in reversed(range(self.n_layers - 1)):
            # Compute gradients
            dw = np.dot(self.cache['A'][i].T, dz) / m
            db = np.mean(dz, axis=0, keepdims=True)
            
            # Add regularization to weight gradients
            if self.regularization == 'l1':
                dw += self.reg_lambda * np.sign(self.weights[i])
            elif self.regularization == 'l2':
                dw += self.reg_lambda * 2 * self.weights[i]
            
            gradients_w.insert(0, dw)
            gradients_b.insert(0, db)
            
            # Propagate error to previous layer
            if i > 0:
                da_prev = np.dot(dz, self.weights[i].T)
                dz = da_prev * self._activation_function(self.cache[f'Z{i}'], derivative=True)
        
        return gradients_w, gradients_b
    
    def _update_parameters(self, gradients_w, gradients_b):
        """Update parameters using the specified optimizer"""
        if self.optimizer == 'sgd':
            self._sgd_update(gradients_w, gradients_b)
        elif self.optimizer == 'momentum':
            self._momentum_update(gradients_w, gradients_b)
        elif self.optimizer == 'adam':
            self._adam_update(gradients_w, gradients_b)
        elif self.optimizer == 'rmsprop':
            self._rmsprop_update(gradients_w, gradients_b)
        elif self.optimizer == 'adagrad':
            self._adagrad_update(gradients_w, gradients_b)
    
    def _sgd_update(self, gradients_w, gradients_b):
        """Standard SGD parameter update"""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * gradients_w[i]
            self.biases[i] -= self.learning_rate * gradients_b[i]
    
    def _momentum_update(self, gradients_w, gradients_b, beta=0.9):
        """SGD with momentum parameter update"""
        for i in range(len(self.weights)):
            self.momentum_w[i] = beta * self.momentum_w[i] + (1 - beta) * gradients_w[i]
            self.momentum_b[i] = beta * self.momentum_b[i] + (1 - beta) * gradients_b[i]
            
            self.weights[i] -= self.learning_rate * self.momentum_w[i]
            self.biases[i] -= self.learning_rate * self.momentum_b[i]
    
    def _adam_update(self, gradients_w, gradients_b):
        """Adam optimizer parameter update"""
        self.t += 1
        
        for i in range(len(self.weights)):
            # Update biased first and second moment estimates
            self.momentum_w[i] = self.beta1 * self.momentum_w[i] + (1 - self.beta1) * gradients_w[i]
            self.momentum_b[i] = self.beta1 * self.momentum_b[i] + (1 - self.beta1) * gradients_b[i]
            
            self.v_w[i] = self.beta2 * self.v_w[i] + (1 - self.beta2) * (gradients_w[i] ** 2)
            self.v_b[i] = self.beta2 * self.v_b[i] + (1 - self.beta2) * (gradients_b[i] ** 2)
            
            # Bias correction
            m_w_corrected = self.momentum_w[i] / (1 - self.beta1 ** self.t)
            m_b_corrected = self.momentum_b[i] / (1 - self.beta1 ** self.t)
            
            v_w_corrected = self.v_w[i] / (1 - self.beta2 ** self.t)
            v_b_corrected = self.v_b[i] / (1 - self.beta2 ** self.t)
            
            # Update parameters
            self.weights[i] -= self.learning_rate * m_w_corrected / (np.sqrt(v_w_corrected) + self.epsilon)
            self.biases[i] -= self.learning_rate * m_b_corrected / (np.sqrt(v_b_corrected) + self.epsilon)
    
    def _rmsprop_update(self, gradients_w, gradients_b, beta=0.9):
        """RMSprop optimizer parameter update"""
        for i in range(len(self.weights)):
            self.v_w[i] = beta * self.v_w[i] + (1 - beta) * (gradients_w[i] ** 2)
            self.v_b[i] = beta * self.v_b[i] + (1 - beta) * (gradients_b[i] ** 2)
            
            self.weights[i] -= self.learning_rate * gradients_w[i] / (np.sqrt(self.v_w[i]) + self.epsilon)
            self.biases[i] -= self.learning_rate * gradients_b[i] / (np.sqrt(self.v_b[i]) + self.epsilon)
    
    def _adagrad_update(self, gradients_w, gradients_b):
        """AdaGrad optimizer parameter update"""
        for i in range(len(self.weights)):
            self.v_w[i] += gradients_w[i] ** 2
            self.v_b[i] += gradients_b[i] ** 2
            
            self.weights[i] -= self.learning_rate * gradients_w[i] / (np.sqrt(self.v_w[i]) + self.epsilon)
            self.biases[i] -= self.learning_rate * gradients_b[i] / (np.sqrt(self.v_b[i]) + self.epsilon)
    
    def _lr_schedule(self, epoch, schedule_type='constant', **kwargs):
        """Apply learning rate scheduling"""
        if schedule_type == 'step':
            step_size = kwargs.get('step_size', 100)
            gamma = kwargs.get('gamma', 0.1)
            self.learning_rate = self.initial_lr * (gamma ** (epoch // step_size))
            
        elif schedule_type == 'exponential':
            gamma = kwargs.get('gamma', 0.95)
            self.learning_rate = self.initial_lr * (gamma ** epoch)
            
        elif schedule_type == 'cosine':
            T_max = kwargs.get('T_max', 1000)
            eta_min = kwargs.get('eta_min', 0)
            self.learning_rate = eta_min + (self.initial_lr - eta_min) * (1 + np.cos(np.pi * epoch / T_max)) / 2
    
    def fit(self, X_train, y_train, X_val=None, y_val=None, epochs=1000, 
            lr_schedule=None, early_stopping=None, verbose=True):
        """
        Train the MLP with comprehensive monitoring and early stopping
        """
        # Convert labels to one-hot encoding
        if len(y_train.shape) == 1:
            n_classes = len(np.unique(y_train))
            y_train_onehot = np.eye(n_classes)[y_train]
        else:
            y_train_onehot = y_train
            
        if X_val is not None and len(y_val.shape) == 1:
            y_val_onehot = np.eye(n_classes)[y_val]
        elif X_val is not None:
            y_val_onehot = y_val
        
        # Initialize early stopping
        best_val_loss = np.inf
        patience_counter = 0
        best_weights = None
        best_biases = None
        
        if verbose:
            print(f"🏋️  Training MLP: {self.layers}")
            print(f"📊 Optimizer: {self.optimizer}, Activation: {self.activation}")
            print(f"🔧 Regularization: {self.regularization}, Dropout: {self.dropout_rate}")
            print("-" * 60)
        
        for epoch in range(epochs):
            # Learning rate scheduling
            if lr_schedule:
                self._lr_schedule(epoch, **lr_schedule)
            
            # Mini-batch training
            n_samples = X_train.shape[0]
            indices = np.random.permutation(n_samples)
            
            epoch_loss = 0
            epoch_acc = 0
            gradient_norm = 0
            
            for i in range(0, n_samples, self.batch_size):
                batch_indices = indices[i:i + self.batch_size]
                X_batch = X_train[batch_indices]
                y_batch = y_train_onehot[batch_indices]
                
                # Forward pass
                y_pred = self.forward(X_batch, training=True)
                
                # Compute loss
                batch_loss = self._compute_loss(y_batch, y_pred)
                epoch_loss += batch_loss
                
                # Compute accuracy
                batch_acc = accuracy_score(np.argmax(y_batch, axis=1), np.argmax(y_pred, axis=1))
                epoch_acc += batch_acc
                
                # Backward pass
                gradients_w, gradients_b = self.backward(X_batch, y_batch, y_pred)
                
                # Compute gradient norm for monitoring
                grad_norm = sum(np.linalg.norm(gw) for gw in gradients_w)
                gradient_norm += grad_norm
                
                # Update parameters
                self._update_parameters(gradients_w, gradients_b)
            
            # Average metrics over batches
            n_batches = (n_samples + self.batch_size - 1) // self.batch_size
            epoch_loss /= n_batches
            epoch_acc /= n_batches
            gradient_norm /= n_batches
            
            # Store training metrics
            self.history['train_loss'].append(epoch_loss)
            self.history['train_acc'].append(epoch_acc)
            self.history['lr_history'].append(self.learning_rate)
            self.history['gradient_norms'].append(gradient_norm)
            
            # Validation metrics
            if X_val is not None:
                val_pred = self.forward(X_val, training=False)
                val_loss = self._compute_loss(y_val_onehot, val_pred)
                val_acc = accuracy_score(np.argmax(y_val_onehot, axis=1), np.argmax(val_pred, axis=1))
                
                self.history['val_loss'].append(val_loss)
                self.history['val_acc'].append(val_acc)
                
                # Early stopping
                if early_stopping and val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_weights = [w.copy() for w in self.weights]
                    best_biases = [b.copy() for b in self.biases]
                elif early_stopping:
                    patience_counter += 1
                    if patience_counter >= early_stopping['patience']:
                        if verbose:
                            print(f"\\nEarly stopping at epoch {epoch}")
                            print(f"Restoring best weights (val_loss: {best_val_loss:.4f})")
                        self.weights = best_weights
                        self.biases = best_biases
                        break
            
            # Progress reporting
            if verbose and epoch % 100 == 0:
                if X_val is not None:
                    print(f"Epoch {epoch:4d} | Train: Loss={epoch_loss:.4f}, Acc={epoch_acc:.4f} | "
                          f"Val: Loss={val_loss:.4f}, Acc={val_acc:.4f} | LR={self.learning_rate:.6f}")
                else:
                    print(f"Epoch {epoch:4d} | Train: Loss={epoch_loss:.4f}, Acc={epoch_acc:.4f} | "
                          f"LR={self.learning_rate:.6f}")
        
        if verbose:
            print("\\n✅ Training completed!")
        
        return self.history
    
    def predict(self, X):
        """Make predictions on new data"""
        y_pred = self.forward(X, training=False)
        return np.argmax(y_pred, axis=1)
    
    def predict_proba(self, X):
        """Get prediction probabilities"""
        return self.forward(X, training=False)
    
    def get_parameters(self):
        """Return current parameters"""
        return {'weights': self.weights, 'biases': self.biases}

# Initialize and demonstrate the advanced MLP
print("\\n🔧 Initializing Advanced MLP...")
mlp_advanced = AdvancedMLP(
    layers=[2, 64, 32, 2], 
    activation='relu',
    optimizer='adam',
    learning_rate=0.001,
    regularization='l2',
    reg_lambda=0.01,
    dropout_rate=0.1,
    batch_size=32
)

print("✅ Advanced MLP initialized successfully!")
print(f"   • Architecture: {mlp_advanced.layers}")
print(f"   • Parameters: {sum(w.size + b.size for w, b in zip(mlp_advanced.weights, mlp_advanced.biases))}")
print(f"   • Optimizer: {mlp_advanced.optimizer}")
print(f"   • Activation: {mlp_advanced.activation}")
print(f"   • Regularization: {mlp_advanced.regularization} (λ={mlp_advanced.reg_lambda})")
print(f"   • Dropout rate: {mlp_advanced.dropout_rate}")

In [ ]:
# 🔬 COMPREHENSIVE ACTIVATION FUNCTION ANALYSIS & OPTIMIZER COMPARISON
print("\\n🔬 Comprehensive Activation Function Analysis & Optimizer Comparison")
print("=" * 75)

# Create challenging datasets for comprehensive testing
def create_benchmark_datasets():
    """Create diverse datasets to test MLP performance"""
    datasets = {}
    
    # 1. Linearly separable data
    X1, y1 = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                                n_informative=2, n_clusters_per_class=1,
                                class_sep=1.5, random_state=42)
    datasets['Linear Separable'] = (X1, y1)
    
    # 2. Non-linear circular pattern
    X2, y2 = make_circles(n_samples=1000, noise=0.1, factor=0.3, random_state=42)
    datasets['Circles'] = (X2, y2)
    
    # 3. Non-linear moon pattern
    X3, y3 = make_moons(n_samples=1000, noise=0.15, random_state=42)
    datasets['Moons'] = (X3, y3)
    
    # 4. Complex multi-class pattern
    X4, y4 = make_classification(n_samples=1200, n_features=2, n_redundant=0,
                                n_informative=2, n_clusters_per_class=2, n_classes=3,
                                class_sep=0.8, random_state=42)
    datasets['Multi-class'] = (X4, y4)
    
    return datasets

datasets = create_benchmark_datasets()

print("📊 Created benchmark datasets:")
for name, (X, y) in datasets.items():
    print(f"   • {name}: {X.shape[0]} samples, {len(np.unique(y))} classes")

# 1. ACTIVATION FUNCTION COMPREHENSIVE ANALYSIS
print("\\n🧬 1. ACTIVATION FUNCTION ANALYSIS")
print("-" * 50)

activations_to_test = ['relu', 'sigmoid', 'tanh', 'leaky_relu']
activation_results = {}

# Test each activation function on moons dataset
X_moons, y_moons = datasets['Moons']
X_train, X_val, y_train, y_val = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42)

# Standardize data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Testing activation functions on Moons dataset...")

for activation in activations_to_test:
    print(f"\\n📈 Training MLP with {activation.upper()} activation...")
    
    mlp_act = AdvancedMLP(
        layers=[2, 32, 16, 2],
        activation=activation,
        optimizer='adam',
        learning_rate=0.01,
        regularization='l2',
        reg_lambda=0.001,
        dropout_rate=0.1,
        batch_size=32,
        random_state=42
    )
    
    # Train with reduced verbosity
    history = mlp_act.fit(X_train_scaled, y_train, X_val_scaled, y_val, 
                         epochs=500, verbose=False)
    
    # Evaluate performance
    train_pred = mlp_act.predict(X_train_scaled)
    val_pred = mlp_act.predict(X_val_scaled)
    
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    
    # Calculate convergence metrics
    final_train_loss = history['train_loss'][-1]
    final_val_loss = history['val_loss'][-1]
    best_val_loss = min(history['val_loss'])
    
    # Find convergence epoch (when 95% of final accuracy is reached)
    target_acc = val_acc * 0.95
    convergence_epoch = len(history['val_acc'])
    for epoch, acc in enumerate(history['val_acc']):
        if acc >= target_acc:
            convergence_epoch = epoch
            break
    
    activation_results[activation] = {
        'train_acc': train_acc,
        'val_acc': val_acc,
        'final_train_loss': final_train_loss,
        'final_val_loss': final_val_loss,
        'best_val_loss': best_val_loss,
        'convergence_epoch': convergence_epoch,
        'history': history,
        'model': mlp_act
    }
    
    print(f"✅ {activation.upper()}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}, Convergence={convergence_epoch} epochs")

# Visualize activation function comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Activation function shapes
ax1 = axes[0, 0]
x = np.linspace(-5, 5, 1000)

# Create dummy MLP for each activation to get function values
for activation in activations_to_test:
    mlp_dummy = AdvancedMLP([1, 1], activation=activation)
    y_act = mlp_dummy._activation_function(x)
    ax1.plot(x, y_act, label=activation.upper(), linewidth=2)

ax1.set_xlabel('Input (z)')
ax1.set_ylabel('Output')
ax1.set_title('Activation Function Shapes', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Training convergence comparison
ax2 = axes[0, 1]
for activation, results in activation_results.items():
    epochs = range(len(results['history']['val_loss']))
    ax2.plot(epochs, results['history']['val_loss'], 
            label=f"{activation.upper()}", linewidth=2, alpha=0.8)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Loss')
ax2.set_title('Training Convergence by Activation', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# Plot 3: Accuracy comparison
ax3 = axes[0, 2]
activations = list(activation_results.keys())
train_accs = [activation_results[act]['train_acc'] for act in activations]
val_accs = [activation_results[act]['val_acc'] for act in activations]

x_pos = np.arange(len(activations))
width = 0.35

ax3.bar(x_pos - width/2, train_accs, width, label='Train Acc', alpha=0.8)
ax3.bar(x_pos + width/2, val_accs, width, label='Val Acc', alpha=0.8)

ax3.set_xlabel('Activation Function')
ax3.set_ylabel('Accuracy')
ax3.set_title('Accuracy Comparison', fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels([act.upper() for act in activations], rotation=45)
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Convergence speed analysis
ax4 = axes[1, 0]
convergence_epochs = [activation_results[act]['convergence_epoch'] for act in activations]

bars = ax4.bar(range(len(activations)), convergence_epochs, alpha=0.8, color=['red', 'blue', 'green', 'orange'])
ax4.set_xlabel('Activation Function')
ax4.set_ylabel('Epochs to 95% Final Accuracy')
ax4.set_title('Convergence Speed', fontweight='bold')
ax4.set_xticks(range(len(activations)))
ax4.set_xticklabels([act.upper() for act in activations], rotation=45)
ax4.grid(True, alpha=0.3)

# Add value labels on bars
for bar, epoch in zip(bars, convergence_epochs):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{epoch}', ha='center', va='bottom', fontweight='bold')

# Plot 5: Decision boundaries for each activation
for i, (activation, results) in enumerate(activation_results.items()):
    ax = axes[1, 1] if i < 2 else axes[1, 2]
    if i >= 2:
        i -= 2
    
    model = results['model']
    
    # Create mesh for decision boundary
    h = 0.02
    x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
    y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    mesh_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(mesh_points)
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    contour = ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    scatter = ax.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], 
                        c=y_train, cmap='RdYlBu', edgecolors='black', s=30)
    
    ax.set_title(f'{activation.upper()} Decision Boundary\\nVal Acc: {results["val_acc"]:.3f}', 
                fontweight='bold')
    ax.grid(True, alpha=0.3)

# Remove extra subplot
if len(activation_results) % 2 == 0:
    fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

# 2. OPTIMIZER COMPARISON
print("\\n⚡ 2. COMPREHENSIVE OPTIMIZER COMPARISON")
print("-" * 50)

optimizers_to_test = ['sgd', 'momentum', 'adam', 'rmsprop', 'adagrad']
optimizer_results = {}

print("Testing optimizers on Circles dataset...")

X_circles, y_circles = datasets['Circles']
X_train, X_val, y_train, y_val = train_test_split(X_circles, y_circles, test_size=0.3, random_state=42)

# Standardize data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

for optimizer in optimizers_to_test:
    print(f"\\n🚀 Training MLP with {optimizer.upper()} optimizer...")
    
    # Adjust learning rate for different optimizers
    lr = 0.01 if optimizer in ['adam', 'rmsprop', 'adagrad'] else 0.1
    
    mlp_opt = AdvancedMLP(
        layers=[2, 32, 16, 2],
        activation='relu',
        optimizer=optimizer,
        learning_rate=lr,
        regularization='l2',
        reg_lambda=0.001,
        dropout_rate=0.05,
        batch_size=32,
        random_state=42
    )
    
    # Train with learning rate scheduling for some optimizers
    lr_schedule = None
    if optimizer == 'sgd':
        lr_schedule = {'schedule_type': 'step', 'step_size': 200, 'gamma': 0.5}
    
    history = mlp_opt.fit(X_train_scaled, y_train, X_val_scaled, y_val,
                         epochs=600, lr_schedule=lr_schedule, verbose=False)
    
    # Evaluate performance
    train_pred = mlp_opt.predict(X_train_scaled)
    val_pred = mlp_opt.predict(X_val_scaled)
    
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    
    # Training stability (variance in last 100 epochs)
    recent_val_losses = history['val_loss'][-100:]
    stability = np.std(recent_val_losses)
    
    optimizer_results[optimizer] = {
        'train_acc': train_acc,
        'val_acc': val_acc,
        'final_val_loss': history['val_loss'][-1],
        'best_val_loss': min(history['val_loss']),
        'stability': stability,
        'history': history,
        'model': mlp_opt
    }
    
    print(f"✅ {optimizer.upper()}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}, Stability={stability:.4f}")

# Visualize optimizer comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Training loss comparison
ax1 = axes[0, 0]
for optimizer, results in optimizer_results.items():
    epochs = range(len(results['history']['train_loss']))
    ax1.plot(epochs, results['history']['train_loss'], 
            label=optimizer.upper(), linewidth=2, alpha=0.8)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss by Optimizer', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Plot 2: Validation accuracy comparison
ax2 = axes[0, 1]
for optimizer, results in optimizer_results.items():
    epochs = range(len(results['history']['val_acc']))
    ax2.plot(epochs, results['history']['val_acc'], 
            label=optimizer.upper(), linewidth=2, alpha=0.8)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Validation Accuracy by Optimizer', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Final performance comparison
ax3 = axes[0, 2]
optimizers = list(optimizer_results.keys())
final_val_accs = [optimizer_results[opt]['val_acc'] for opt in optimizers]

bars = ax3.bar(range(len(optimizers)), final_val_accs, alpha=0.8, 
               color=['red', 'blue', 'green', 'orange', 'purple'])
ax3.set_xlabel('Optimizer')
ax3.set_ylabel('Final Validation Accuracy')
ax3.set_title('Final Performance Comparison', fontweight='bold')
ax3.set_xticks(range(len(optimizers)))
ax3.set_xticklabels([opt.upper() for opt in optimizers], rotation=45)
ax3.grid(True, alpha=0.3)

# Add value labels
for bar, acc in zip(bars, final_val_accs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

# Plot 4: Training stability analysis
ax4 = axes[1, 0]
stabilities = [optimizer_results[opt]['stability'] for opt in optimizers]

bars = ax4.bar(range(len(optimizers)), stabilities, alpha=0.8,
               color=['red', 'blue', 'green', 'orange', 'purple'])
ax4.set_xlabel('Optimizer')
ax4.set_ylabel('Training Stability (Loss Std)')
ax4.set_title('Training Stability Analysis', fontweight='bold')
ax4.set_xticks(range(len(optimizers)))
ax4.set_xticklabels([opt.upper() for opt in optimizers], rotation=45)
ax4.grid(True, alpha=0.3)

# Plot 5: Learning rate evolution for adaptive optimizers
ax5 = axes[1, 1]
for optimizer, results in optimizer_results.items():
    if 'lr_history' in results['history'] and len(results['history']['lr_history']) > 0:
        epochs = range(len(results['history']['lr_history']))
        ax5.plot(epochs, results['history']['lr_history'], 
                label=optimizer.upper(), linewidth=2, alpha=0.8)

ax5.set_xlabel('Epoch')
ax5.set_ylabel('Learning Rate')
ax5.set_title('Learning Rate Evolution', fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)
ax5.set_yscale('log')

# Plot 6: Gradient norm evolution
ax6 = axes[1, 2]
for optimizer, results in optimizer_results.items():
    if 'gradient_norms' in results['history'] and len(results['history']['gradient_norms']) > 0:
        epochs = range(len(results['history']['gradient_norms']))
        ax6.plot(epochs, results['history']['gradient_norms'], 
                label=optimizer.upper(), linewidth=2, alpha=0.8)

ax6.set_xlabel('Epoch')
ax6.set_ylabel('Gradient Norm')
ax6.set_title('Gradient Norm Evolution', fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)
ax6.set_yscale('log')

plt.tight_layout()
plt.show()

# Summary analysis
print("\\n📋 ANALYSIS SUMMARY")
print("=" * 40)

print("\\n🧬 Activation Function Results:")
activation_df = pd.DataFrame({
    'Activation': [act.upper() for act in activations_to_test],
    'Train Acc': [f"{activation_results[act]['train_acc']:.3f}" for act in activations_to_test],
    'Val Acc': [f"{activation_results[act]['val_acc']:.3f}" for act in activations_to_test],
    'Convergence': [f"{activation_results[act]['convergence_epoch']}" for act in activations_to_test],
    'Final Val Loss': [f"{activation_results[act]['final_val_loss']:.4f}" for act in activations_to_test]
})
print(activation_df.to_string(index=False))

best_activation = max(activation_results.items(), key=lambda x: x[1]['val_acc'])
print(f"\\n🏆 Best Activation: {best_activation[0].upper()} (Val Acc: {best_activation[1]['val_acc']:.3f})")

print("\\n⚡ Optimizer Results:")
optimizer_df = pd.DataFrame({
    'Optimizer': [opt.upper() for opt in optimizers_to_test],
    'Train Acc': [f"{optimizer_results[opt]['train_acc']:.3f}" for opt in optimizers_to_test],
    'Val Acc': [f"{optimizer_results[opt]['val_acc']:.3f}" for opt in optimizers_to_test],
    'Stability': [f"{optimizer_results[opt]['stability']:.4f}" for opt in optimizers_to_test],
    'Best Val Loss': [f"{optimizer_results[opt]['best_val_loss']:.4f}" for opt in optimizers_to_test]
})
print(optimizer_df.to_string(index=False))

best_optimizer = max(optimizer_results.items(), key=lambda x: x[1]['val_acc'])
print(f"\\n🏆 Best Optimizer: {best_optimizer[0].upper()} (Val Acc: {best_optimizer[1]['val_acc']:.3f})")

print("\\n💡 Key Insights:")
print("• ReLU generally provides fastest convergence and good performance")
print("• Adam optimizer typically offers the best balance of speed and stability")
print("• Leaky ReLU can help prevent dead neuron problems")
print("• SGD with momentum can be competitive with proper learning rate scheduling")
print("• AdaGrad may suffer from aggressive learning rate decay")

In [ ]:
# 🚀 REAL-WORLD APPLICATIONS AND COMPREHENSIVE BENCHMARKING
print("\\n🚀 Real-World Applications and Comprehensive Benchmarking")
print("=" * 70)

from sklearn.datasets import load_digits, load_wine, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import time

# Real-world application 1: Handwritten Digit Recognition
print("\\n🔢 Application 1: Handwritten Digit Recognition (8x8 Images)")
print("-" * 60)

# Load and prepare digits dataset
digits = load_digits()
X_digits, y_digits = digits.data, digits.target

print(f"Dataset: {X_digits.shape[0]} samples, {X_digits.shape[1]} features, {len(np.unique(y_digits))} classes")
print("Task: Multi-class classification of handwritten digits (0-9)")

# Split and scale data
X_train_digits, X_test_digits, y_train_digits, y_test_digits = train_test_split(
    X_digits, y_digits, test_size=0.2, stratify=y_digits, random_state=42)

scaler_digits = StandardScaler()
X_train_digits_scaled = scaler_digits.fit_transform(X_train_digits)
X_test_digits_scaled = scaler_digits.transform(X_test_digits)

print(f"Training set: {X_train_digits_scaled.shape[0]} samples")
print(f"Test set: {X_test_digits_scaled.shape[0]} samples")

# Train optimized MLP for digits
print("\\n🧠 Training Optimized MLP for Digit Recognition...")
mlp_digits = AdvancedMLP(
    layers=[64, 128, 64, 32, 10],  # Deeper architecture for complex task
    activation='relu',
    optimizer='adam',
    learning_rate=0.001,
    regularization='l2',
    reg_lambda=0.0001,
    dropout_rate=0.2,
    batch_size=64,
    random_state=42
)

# Train with early stopping
early_stopping = {'patience': 50}
digits_history = mlp_digits.fit(
    X_train_digits_scaled, y_train_digits, 
    X_test_digits_scaled, y_test_digits,
    epochs=800,
    early_stopping=early_stopping,
    verbose=True
)

# Evaluate on digits
digits_pred = mlp_digits.predict(X_test_digits_scaled)
digits_accuracy = accuracy_score(y_test_digits, digits_pred)

print(f"\\n✅ Digits MLP Test Accuracy: {digits_accuracy:.4f}")

# Real-world application 2: Wine Quality Classification
print("\\n🍷 Application 2: Wine Quality Classification")
print("-" * 50)

# Load wine dataset
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

print(f"Dataset: {X_wine.shape[0]} samples, {X_wine.shape[1]} features, {len(np.unique(y_wine))} classes")
print("Task: Wine quality classification based on chemical properties")

# Split and scale data
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.3, stratify=y_wine, random_state=42)

scaler_wine = StandardScaler()
X_train_wine_scaled = scaler_wine.fit_transform(X_train_wine)
X_test_wine_scaled = scaler_wine.transform(X_test_wine)

# Train MLP for wine classification
print("\\n🧠 Training MLP for Wine Classification...")
mlp_wine = AdvancedMLP(
    layers=[13, 32, 16, 3],
    activation='relu',
    optimizer='adam',
    learning_rate=0.01,
    regularization='l2',
    reg_lambda=0.01,
    dropout_rate=0.1,
    batch_size=16,
    random_state=42
)

wine_history = mlp_wine.fit(
    X_train_wine_scaled, y_train_wine,
    X_test_wine_scaled, y_test_wine,
    epochs=500,
    verbose=False
)

wine_pred = mlp_wine.predict(X_test_wine_scaled)
wine_accuracy = accuracy_score(y_test_wine, wine_pred)

print(f"✅ Wine MLP Test Accuracy: {wine_accuracy:.4f}")

# Real-world application 3: Breast Cancer Diagnosis
print("\\n🏥 Application 3: Breast Cancer Diagnosis")
print("-" * 50)

# Load breast cancer dataset
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

print(f"Dataset: {X_cancer.shape[0]} samples, {X_cancer.shape[1]} features, {len(np.unique(y_cancer))} classes")
print("Task: Binary classification of breast cancer (malignant/benign)")

# Split and scale data
X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(
    X_cancer, y_cancer, test_size=0.2, stratify=y_cancer, random_state=42)

scaler_cancer = StandardScaler()
X_train_cancer_scaled = scaler_cancer.fit_transform(X_train_cancer)
X_test_cancer_scaled = scaler_cancer.transform(X_test_cancer)

# Train MLP for cancer diagnosis
print("\\n🧠 Training MLP for Cancer Diagnosis...")
mlp_cancer = AdvancedMLP(
    layers=[30, 16, 8, 2],
    activation='relu',
    optimizer='adam',
    learning_rate=0.005,
    regularization='l2',
    reg_lambda=0.001,
    dropout_rate=0.1,
    batch_size=32,
    random_state=42
)

cancer_history = mlp_cancer.fit(
    X_train_cancer_scaled, y_train_cancer,
    X_test_cancer_scaled, y_test_cancer,
    epochs=400,
    verbose=False
)

cancer_pred = mlp_cancer.predict(X_test_cancer_scaled)
cancer_accuracy = accuracy_score(y_test_cancer, cancer_pred)

print(f"✅ Cancer MLP Test Accuracy: {cancer_accuracy:.4f}")

# COMPREHENSIVE BENCHMARKING AGAINST OTHER ALGORITHMS
print("\\n📊 Comprehensive Benchmarking Against Other ML Algorithms")
print("=" * 65)

# Define algorithms for comparison
algorithms = {
    'MLP (Ours)': None,  # Will be filled with our results
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

# Datasets for benchmarking
benchmark_datasets = {
    'Digits': (X_train_digits_scaled, X_test_digits_scaled, y_train_digits, y_test_digits),
    'Wine': (X_train_wine_scaled, X_test_wine_scaled, y_train_wine, y_test_wine),
    'Breast Cancer': (X_train_cancer_scaled, X_test_cancer_scaled, y_train_cancer, y_test_cancer)
}

# MLP results from previous training
mlp_results = {
    'Digits': digits_accuracy,
    'Wine': wine_accuracy,
    'Breast Cancer': cancer_accuracy
}

# Benchmark results storage
benchmark_results = {}

print("\\nRunning comprehensive benchmarking...")
print("-" * 40)

for dataset_name, (X_train, X_test, y_train, y_test) in benchmark_datasets.items():
    print(f"\\n📈 Benchmarking on {dataset_name} dataset:")
    
    dataset_results = {}
    
    # Add our MLP results
    dataset_results['MLP (Ours)'] = {
        'accuracy': mlp_results[dataset_name],
        'train_time': 0.0  # Already trained
    }
    
    # Test other algorithms
    for alg_name, algorithm in algorithms.items():
        if alg_name == 'MLP (Ours)':
            continue
            
        print(f"   Testing {alg_name}...")
        
        start_time = time.time()
        algorithm.fit(X_train, y_train)
        train_time = time.time() - start_time
        
        y_pred = algorithm.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        dataset_results[alg_name] = {
            'accuracy': accuracy,
            'train_time': train_time
        }
        
        print(f"     ✅ {alg_name}: Accuracy = {accuracy:.4f}, Time = {train_time:.3f}s")
    
    benchmark_results[dataset_name] = dataset_results

# Visualization of benchmarking results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy comparison across datasets
ax1 = axes[0, 0]
algorithms_list = list(algorithms.keys())
datasets_list = list(benchmark_datasets.keys())

# Create accuracy matrix
accuracy_matrix = np.zeros((len(algorithms_list), len(datasets_list)))
for i, alg in enumerate(algorithms_list):
    for j, dataset in enumerate(datasets_list):
        accuracy_matrix[i, j] = benchmark_results[dataset][alg]['accuracy']

# Create heatmap
im = ax1.imshow(accuracy_matrix, cmap='RdYlGn', aspect='auto', vmin=0.8, vmax=1.0)
ax1.set_xticks(range(len(datasets_list)))
ax1.set_xticklabels(datasets_list)
ax1.set_yticks(range(len(algorithms_list)))
ax1.set_yticklabels(algorithms_list)
ax1.set_title('Algorithm Performance Heatmap\\n(Test Accuracy)', fontweight='bold')

# Add text annotations
for i in range(len(algorithms_list)):
    for j in range(len(datasets_list)):
        text = ax1.text(j, i, f'{accuracy_matrix[i, j]:.3f}',
                       ha="center", va="center", color="black", fontweight='bold')

plt.colorbar(im, ax=ax1, label='Accuracy')

# Plot 2: Training time comparison
ax2 = axes[0, 1]
alg_names = [alg for alg in algorithms_list if alg != 'MLP (Ours)']  # Exclude MLP for time comparison
datasets_times = {dataset: [] for dataset in datasets_list}

for dataset in datasets_list:
    for alg in alg_names:
        datasets_times[dataset].append(benchmark_results[dataset][alg]['train_time'])

x = np.arange(len(alg_names))
width = 0.25

for i, dataset in enumerate(datasets_list):
    ax2.bar(x + i*width, datasets_times[dataset], width, label=dataset, alpha=0.8)

ax2.set_xlabel('Algorithm')
ax2.set_ylabel('Training Time (seconds)')
ax2.set_title('Training Time Comparison', fontweight='bold')
ax2.set_xticks(x + width)
ax2.set_xticklabels([alg.replace(' ', '\\n') for alg in alg_names], rotation=45, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# Plot 3: Algorithm ranking by dataset
ax3 = axes[1, 0]

# Calculate average rank for each algorithm
algorithm_ranks = {}
for alg in algorithms_list:
    ranks = []
    for dataset in datasets_list:
        # Get accuracies for this dataset and sort them
        dataset_accs = [(name, benchmark_results[dataset][name]['accuracy']) 
                       for name in algorithms_list]
        dataset_accs.sort(key=lambda x: x[1], reverse=True)
        
        # Find rank of current algorithm
        for rank, (name, _) in enumerate(dataset_accs):
            if name == alg:
                ranks.append(rank + 1)
                break
    
    algorithm_ranks[alg] = np.mean(ranks)

# Sort algorithms by average rank
sorted_algorithms = sorted(algorithm_ranks.items(), key=lambda x: x[1])
alg_names_sorted, avg_ranks = zip(*sorted_algorithms)

bars = ax3.barh(range(len(alg_names_sorted)), avg_ranks, alpha=0.8, 
                color=['gold' if 'MLP' in alg else 'lightblue' for alg in alg_names_sorted])
ax3.set_yticks(range(len(alg_names_sorted)))
ax3.set_yticklabels(alg_names_sorted)
ax3.set_xlabel('Average Rank (1=Best)')
ax3.set_title('Algorithm Ranking\\n(Lower is Better)', fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add value labels
for bar, rank in zip(bars, avg_ranks):
    ax3.text(rank + 0.05, bar.get_y() + bar.get_height()/2, 
            f'{rank:.1f}', ha='left', va='center', fontweight='bold')

# Plot 4: Performance vs Complexity Analysis
ax4 = axes[1, 1]

# Calculate average accuracy and complexity (training time) for each algorithm
avg_accuracies = []
avg_times = []
alg_labels = []

for alg in algorithms_list:
    if alg == 'MLP (Ours)':
        continue  # Skip MLP for training time analysis
    
    accs = [benchmark_results[dataset][alg]['accuracy'] for dataset in datasets_list]
    times = [benchmark_results[dataset][alg]['train_time'] for dataset in datasets_list]
    
    avg_accuracies.append(np.mean(accs))
    avg_times.append(np.mean(times))
    alg_labels.append(alg)

# Add MLP with estimated time
mlp_avg_acc = np.mean([benchmark_results[dataset]['MLP (Ours)']['accuracy'] for dataset in datasets_list])
avg_accuracies.append(mlp_avg_acc)
avg_times.append(2.0)  # Estimated average training time
alg_labels.append('MLP (Ours)')

# Create scatter plot
colors = ['red' if 'MLP' in alg else 'blue' for alg in alg_labels]
sizes = [100 if 'MLP' in alg else 60 for alg in alg_labels]

scatter = ax4.scatter(avg_times, avg_accuracies, c=colors, s=sizes, alpha=0.7, edgecolors='black')

# Add labels
for i, alg in enumerate(alg_labels):
    ax4.annotate(alg.replace(' ', '\\n'), (avg_times[i], avg_accuracies[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9,
                ha='left', va='bottom')

ax4.set_xlabel('Average Training Time (seconds)')
ax4.set_ylabel('Average Accuracy')
ax4.set_title('Performance vs Training Time\\n(Bigger=MLP)', fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.set_xscale('log')

plt.tight_layout()
plt.show()

# Create comprehensive results table
print("\\n📋 COMPREHENSIVE BENCHMARKING RESULTS")
print("=" * 50)

# Create detailed results table
results_data = []
for dataset in datasets_list:
    for alg in algorithms_list:
        results_data.append({
            'Dataset': dataset,
            'Algorithm': alg,
            'Accuracy': f"{benchmark_results[dataset][alg]['accuracy']:.4f}",
            'Training Time': f"{benchmark_results[dataset][alg]['train_time']:.3f}s" if benchmark_results[dataset][alg]['train_time'] > 0 else 'Pre-trained'
        })

results_df = pd.DataFrame(results_data)
print(results_df.to_string(index=False))

# Summary statistics
print("\\n🏆 PERFORMANCE SUMMARY")
print("=" * 30)

# Find best algorithm for each dataset
for dataset in datasets_list:
    best_alg = max(benchmark_results[dataset].items(), key=lambda x: x[1]['accuracy'])
    print(f"• {dataset}: {best_alg[0]} (Accuracy: {best_alg[1]['accuracy']:.4f})")

# Overall champion
overall_scores = {}
for alg in algorithms_list:
    scores = [benchmark_results[dataset][alg]['accuracy'] for dataset in datasets_list]
    overall_scores[alg] = np.mean(scores)

champion = max(overall_scores.items(), key=lambda x: x[1])
print(f"\\n🥇 Overall Champion: {champion[0]} (Avg Accuracy: {champion[1]:.4f})")

print("\\n💡 KEY INSIGHTS FROM BENCHMARKING:")
print("=" * 45)
print("• MLP shows competitive performance across diverse datasets")
print("• Random Forest provides excellent baseline performance")
print("• SVM excels on smaller, high-dimensional datasets")
print("• KNN works well with sufficient training data")
print("• MLP benefits from proper hyperparameter tuning")
print("• Training time varies significantly across algorithms")
print("• No single algorithm dominates all problem types")

In [ ]:
# 🎨 INTERACTIVE VISUALIZATIONS & ADVANCED TRAINING DYNAMICS
print("\\n🎨 Interactive Visualizations & Advanced Training Dynamics")
print("=" * 70)

# Training Dynamics Visualization Functions
def plot_comprehensive_training_analysis(history, model_name="MLP"):
    """Create comprehensive training analysis with multiple diagnostic plots"""
    
    fig, axes = plt.subplots(3, 3, figsize=(20, 18))
    
    epochs = range(len(history['train_loss']))
    
    # Plot 1: Loss Evolution with Smoothing
    ax1 = axes[0, 0]
    ax1.plot(epochs, history['train_loss'], alpha=0.3, color='blue', label='Train Loss (Raw)')
    ax1.plot(epochs, history['val_loss'], alpha=0.3, color='red', label='Val Loss (Raw)')
    
    # Add smoothed curves
    if len(epochs) > 10:
        window = max(5, len(epochs) // 20)
        train_smooth = pd.Series(history['train_loss']).rolling(window=window, center=True).mean()
        val_smooth = pd.Series(history['val_loss']).rolling(window=window, center=True).mean()
        ax1.plot(epochs, train_smooth, color='blue', linewidth=3, label='Train (Smoothed)')
        ax1.plot(epochs, val_smooth, color='red', linewidth=3, label='Val (Smoothed)')
    
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{model_name}: Training Loss Evolution', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')
    
    # Plot 2: Accuracy Evolution
    ax2 = axes[0, 1]
    ax2.plot(epochs, history['train_acc'], color='blue', linewidth=2, label='Train Accuracy')
    ax2.plot(epochs, history['val_acc'], color='red', linewidth=2, label='Val Accuracy')
    
    # Mark best validation accuracy
    best_val_idx = np.argmax(history['val_acc'])
    ax2.axvline(x=best_val_idx, color='green', linestyle='--', alpha=0.7)
    ax2.scatter([best_val_idx], [history['val_acc'][best_val_idx]], 
               color='green', s=100, zorder=5, label=f'Best Val (Epoch {best_val_idx})')
    
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{model_name}: Accuracy Evolution', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Learning Rate Schedule
    ax3 = axes[0, 2]
    if 'lr_history' in history and len(history['lr_history']) > 0:
        ax3.plot(epochs, history['lr_history'], color='orange', linewidth=2)
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Learning Rate')
        ax3.set_title(f'{model_name}: Learning Rate Schedule', fontweight='bold')
        ax3.grid(True, alpha=0.3)
        ax3.set_yscale('log')
    else:
        ax3.text(0.5, 0.5, 'Learning Rate\\nHistory\\nNot Available', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=14)
        ax3.set_title('Learning Rate Schedule', fontweight='bold')
    
    # Plot 4: Gradient Norm Evolution
    ax4 = axes[1, 0]
    if 'gradient_norms' in history and len(history['gradient_norms']) > 0:
        ax4.plot(epochs, history['gradient_norms'], color='purple', linewidth=2, alpha=0.7)
        
        # Add smoothed gradient norm
        if len(epochs) > 10:
            grad_smooth = pd.Series(history['gradient_norms']).rolling(window=window, center=True).mean()
            ax4.plot(epochs, grad_smooth, color='purple', linewidth=3, label='Smoothed')
            ax4.legend()
        
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Gradient Norm')
        ax4.set_title(f'{model_name}: Gradient Flow', fontweight='bold')
        ax4.grid(True, alpha=0.3)
        ax4.set_yscale('log')
    else:
        ax4.text(0.5, 0.5, 'Gradient Norm\\nHistory\\nNot Available', 
                ha='center', va='center', transform=ax4.transAxes, fontsize=14)
        ax4.set_title('Gradient Flow', fontweight='bold')
    
    # Plot 5: Overfitting Analysis
    ax5 = axes[1, 1]
    
    # Calculate generalization gap
    gap = np.array(history['train_acc']) - np.array(history['val_acc'])
    ax5.plot(epochs, gap, color='red', linewidth=2, label='Generalization Gap')
    ax5.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Add trend line
    if len(epochs) > 20:
        z = np.polyfit(epochs, gap, 1)
        p = np.poly1d(z)
        ax5.plot(epochs, p(epochs), color='red', linestyle='--', alpha=0.7, label='Trend')
    
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('Train Acc - Val Acc')
    ax5.set_title(f'{model_name}: Overfitting Analysis', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Training Speed Analysis
    ax6 = axes[1, 2]
    
    # Loss improvement rate
    train_loss_diff = np.diff(history['train_loss'])
    val_loss_diff = np.diff(history['val_loss'])
    
    ax6.plot(epochs[1:], -train_loss_diff, color='blue', alpha=0.7, label='Train Loss Improvement')
    ax6.plot(epochs[1:], -val_loss_diff, color='red', alpha=0.7, label='Val Loss Improvement')
    ax6.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    ax6.set_xlabel('Epoch')
    ax6.set_ylabel('Loss Improvement')
    ax6.set_title(f'{model_name}: Learning Speed', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # Plot 7: Loss Landscape (Train vs Val)
    ax7 = axes[2, 0]
    scatter = ax7.scatter(history['train_loss'], history['val_loss'], 
                         c=epochs, cmap='viridis', alpha=0.7, s=30)
    
    # Add diagonal line (perfect generalization)
    min_loss = min(min(history['train_loss']), min(history['val_loss']))
    max_loss = max(max(history['train_loss']), max(history['val_loss']))
    ax7.plot([min_loss, max_loss], [min_loss, max_loss], 'k--', alpha=0.5, label='Perfect Generalization')
    
    ax7.set_xlabel('Training Loss')
    ax7.set_ylabel('Validation Loss')
    ax7.set_title(f'{model_name}: Loss Landscape', fontweight='bold')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax7, label='Epoch')
    
    # Plot 8: Training Stability Analysis
    ax8 = axes[2, 1]
    
    # Calculate rolling standard deviation of validation loss
    if len(epochs) > 20:
        window = min(20, len(epochs) // 5)
        val_loss_std = pd.Series(history['val_loss']).rolling(window=window).std()
        ax8.plot(epochs, val_loss_std, color='red', linewidth=2, label='Val Loss Stability')
        
        # Add moving average
        val_loss_ma = pd.Series(history['val_loss']).rolling(window=window).mean()
        ax8_twin = ax8.twinx()
        ax8_twin.plot(epochs, val_loss_ma, color='blue', linewidth=2, alpha=0.5, label='Val Loss MA')
        ax8_twin.set_ylabel('Validation Loss (MA)', color='blue')
        ax8_twin.tick_params(axis='y', labelcolor='blue')
        
        ax8.set_xlabel('Epoch')
        ax8.set_ylabel('Val Loss Std Dev', color='red')
        ax8.tick_params(axis='y', labelcolor='red')
        ax8.set_title(f'{model_name}: Training Stability', fontweight='bold')
        ax8.grid(True, alpha=0.3)
        
        # Add stability assessment
        recent_std = np.mean(val_loss_std[-10:]) if len(val_loss_std) >= 10 else np.nan
        if not np.isnan(recent_std):
            stability_text = "Stable" if recent_std < 0.01 else "Unstable" if recent_std > 0.05 else "Moderate"
            ax8.text(0.05, 0.95, f'Recent Stability: {stability_text}', transform=ax8.transAxes,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
                    verticalalignment='top')
    else:
        ax8.text(0.5, 0.5, 'Insufficient Data\\nfor Stability\\nAnalysis', 
                ha='center', va='center', transform=ax8.transAxes, fontsize=14)
        ax8.set_title('Training Stability', fontweight='bold')
    
    # Plot 9: Performance Summary
    ax9 = axes[2, 2]
    
    # Key metrics summary
    final_train_acc = history['train_acc'][-1]
    final_val_acc = history['val_acc'][-1]
    best_val_acc = max(history['val_acc'])
    final_train_loss = history['train_loss'][-1]
    final_val_loss = history['val_loss'][-1]
    
    # Create summary text
    summary_text = f\"\"\"PERFORMANCE SUMMARY
    
Final Training Accuracy: {final_train_acc:.4f}
Final Validation Accuracy: {final_val_acc:.4f}
Best Validation Accuracy: {best_val_acc:.4f}

Final Training Loss: {final_train_loss:.4f}
Final Validation Loss: {final_val_loss:.4f}

Total Epochs: {len(epochs)}
Best Epoch: {best_val_idx}

Generalization Gap: {final_train_acc - final_val_acc:.4f}
\"\"\"
    
    ax9.text(0.1, 0.9, summary_text, transform=ax9.transAxes, fontsize=11,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))
    
    ax9.set_title(f'{model_name}: Summary', fontweight='bold')
    ax9.axis('off')
    
    plt.tight_layout()
    plt.show()

# Create comprehensive training analysis for our models
print("\\n📊 Comprehensive Training Dynamics Analysis")
print("-" * 55)

# Analyze digits training
print("\\n🔢 Digits Classification Training Analysis:")
plot_comprehensive_training_analysis(digits_history, "Digits MLP")

# Train a new model for moons dataset to show training dynamics
print("\\n🌙 Training MLP on Moons Dataset for Detailed Analysis:")

# Prepare moons data
X_moons_demo, y_moons_demo = make_moons(n_samples=800, noise=0.2, random_state=42)
X_train_demo, X_val_demo, y_train_demo, y_val_demo = train_test_split(
    X_moons_demo, y_moons_demo, test_size=0.3, random_state=42)

scaler_demo = StandardScaler()
X_train_demo_scaled = scaler_demo.fit_transform(X_train_demo)
X_val_demo_scaled = scaler_demo.transform(X_val_demo)

# Train MLP with comprehensive tracking
mlp_demo = AdvancedMLP(
    layers=[2, 32, 16, 8, 2],
    activation='relu',
    optimizer='adam',
    learning_rate=0.01,
    regularization='l2',
    reg_lambda=0.001,
    dropout_rate=0.1,
    batch_size=32,
    random_state=42
)

# Train with learning rate scheduling
lr_schedule = {'schedule_type': 'cosine', 'T_max': 500, 'eta_min': 0.0001}
demo_history = mlp_demo.fit(
    X_train_demo_scaled, y_train_demo,
    X_val_demo_scaled, y_val_demo,
    epochs=500,
    lr_schedule=lr_schedule,
    verbose=False
)

print("Training completed. Generating comprehensive analysis...")

# Analyze demo training
plot_comprehensive_training_analysis(demo_history, "Moons MLP")

# DECISION BOUNDARY EVOLUTION VISUALIZATION
print("\\n🎯 Decision Boundary Evolution Analysis")
print("-" * 45)

def plot_decision_boundary_evolution(model, X_data, y_data, scaler, model_name="MLP"):
    """Plot how decision boundary evolves during training"""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Create mesh for decision boundary
    h = 0.02
    x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
    y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    mesh_points = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    
    # Final decision boundary
    ax = axes[0, 0]
    Z = model.predict(mesh_points)
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    scatter = ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='RdYlBu', edgecolors='black')
    ax.set_title(f'{model_name}: Final Decision Boundary', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Prediction confidence map
    ax = axes[0, 1]
    Z_prob = model.predict_proba(mesh_points)
    Z_confidence = np.max(Z_prob, axis=1).reshape(xx.shape)
    
    contour = ax.contourf(xx, yy, Z_confidence, levels=20, alpha=0.6, cmap='viridis')
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='RdYlBu', edgecolors='black')
    ax.set_title(f'{model_name}: Prediction Confidence', fontweight='bold')
    plt.colorbar(contour, ax=ax, label='Max Probability')
    ax.grid(True, alpha=0.3)
    
    # Class probability heatmaps
    for class_idx in range(2):
        ax = axes[0, 2] if class_idx == 0 else axes[1, 0]
        Z_class = Z_prob[:, class_idx].reshape(xx.shape)
        
        contour = ax.contourf(xx, yy, Z_class, levels=20, alpha=0.6, cmap='Blues')
        ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='RdYlBu', edgecolors='black')
        ax.set_title(f'{model_name}: Class {class_idx} Probability', fontweight='bold')
        plt.colorbar(contour, ax=ax, label=f'P(Class {class_idx})')
        ax.grid(True, alpha=0.3)
    
    # Uncertainty map (entropy)
    ax = axes[1, 1]
    epsilon = 1e-15
    Z_prob_safe = np.clip(Z_prob, epsilon, 1 - epsilon)
    Z_entropy = -np.sum(Z_prob_safe * np.log(Z_prob_safe), axis=1).reshape(xx.shape)
    
    contour = ax.contourf(xx, yy, Z_entropy, levels=20, alpha=0.6, cmap='Reds')
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='RdYlBu', edgecolors='black')
    ax.set_title(f'{model_name}: Prediction Uncertainty (Entropy)', fontweight='bold')
    plt.colorbar(contour, ax=ax, label='Entropy')
    ax.grid(True, alpha=0.3)
    
    # Model architecture visualization
    ax = axes[1, 2]
    
    # Simple network diagram
    layers = model.layers
    max_neurons = max(layers)
    
    # Plot network nodes
    for i, layer_size in enumerate(layers):
        x = i
        y_positions = np.linspace(-max_neurons/2, max_neurons/2, layer_size)
        
        for j, y in enumerate(y_positions):
            circle = plt.Circle((x, y), 0.1, color='lightblue', ec='black')
            ax.add_patch(circle)
            
            # Add connections to next layer
            if i < len(layers) - 1:
                next_layer_positions = np.linspace(-max_neurons/2, max_neurons/2, layers[i+1])
                for next_y in next_layer_positions:
                    ax.plot([x + 0.1, x + 0.9], [y, next_y], 'k-', alpha=0.3, linewidth=0.5)
    
    ax.set_xlim(-0.5, len(layers) - 0.5)
    ax.set_ylim(-max_neurons/2 - 1, max_neurons/2 + 1)
    ax.set_xlabel('Layers')
    ax.set_title(f'{model_name}: Network Architecture\\n{layers}', fontweight='bold')
    
    # Add layer labels
    layer_names = ['Input'] + [f'Hidden {i}' for i in range(1, len(layers)-1)] + ['Output']
    for i, name in enumerate(layer_names):
        ax.text(i, max_neurons/2 + 0.5, f'{name}\\n({layers[i]})', ha='center', va='bottom', fontweight='bold')
    
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Generate decision boundary analysis for our demo model
all_data = np.vstack([X_train_demo, X_val_demo])
all_labels = np.hstack([y_train_demo, y_val_demo])

plot_decision_boundary_evolution(mlp_demo, all_data, all_labels, scaler_demo, "Moons MLP")

# FINAL COMPREHENSIVE INSIGHTS AND RECOMMENDATIONS
print("\\n🎓 COMPREHENSIVE INSIGHTS AND RECOMMENDATIONS")
print("=" * 55)

print("\\n💡 KEY INSIGHTS FROM COMPREHENSIVE ANALYSIS:")
print("-" * 50)
print("✅ MATHEMATICAL FOUNDATIONS:")
print("   • MLPs are universal function approximators")
print("   • Backpropagation efficiently computes gradients using chain rule")
print("   • Non-linear activations are essential for learning complex patterns")

print("\\n✅ OPTIMIZATION INSIGHTS:")
print("   • Adam optimizer generally provides best balance of speed and stability")
print("   • Learning rate scheduling can significantly improve convergence")
print("   • Gradient monitoring helps detect training issues early")

print("\\n✅ REGULARIZATION BENEFITS:")
print("   • L2 regularization prevents overfitting effectively")
print("   • Dropout provides additional generalization benefits")
print("   • Early stopping prevents overtraining")

print("\\n✅ ACTIVATION FUNCTION GUIDELINES:")
print("   • ReLU: Fast, effective, but can suffer from dead neurons")
print("   • Leaky ReLU: Fixes dead neuron problem")
print("   • Sigmoid/Tanh: Smooth but prone to vanishing gradients")

print("\\n🏗️ ARCHITECTURE DESIGN PRINCIPLES:")
print("-" * 40)
print("• Start simple, increase complexity gradually")
print("• Deeper networks can model more complex functions")
print("• Width vs depth: depth usually more parameter-efficient")
print("• Match architecture complexity to problem complexity")

print("\\n⚙️ HYPERPARAMETER TUNING BEST PRACTICES:")
print("-" * 45)
print("• Learning rate: Start with 0.001-0.01 for Adam")
print("• Batch size: 32-128 usually works well")
print("• Regularization: λ = 0.001-0.01 for L2")
print("• Dropout: 0.1-0.3 for hidden layers")
print("• Use validation set for hyperparameter selection")

print("\\n🔧 DEBUGGING TRAINING ISSUES:")
print("-" * 35)
print("• Loss not decreasing: Check learning rate, gradients")
print("• Overfitting: Add regularization, reduce model complexity")
print("• Underfitting: Increase model capacity, reduce regularization")
print("• Unstable training: Lower learning rate, check data preprocessing")

print("\\n🚀 WHEN TO USE MLPs:")
print("-" * 25)
print("✅ Tabular data with complex non-linear relationships")
print("✅ Classification tasks with moderate dataset sizes")
print("✅ When interpretability is less critical than performance")
print("✅ As baseline before trying more complex architectures")

print("\\n⚠️ WHEN TO AVOID MLPs:")
print("-" * 27)
print("❌ Sequential data (use RNNs/Transformers)")
print("❌ Image data (use CNNs)")
print("❌ Very large datasets (consider efficiency)")
print("❌ When simple linear models work well")

print("\\n🎯 NEXT STEPS FOR ADVANCED LEARNING:")
print("-" * 40)
print("• Explore batch normalization for deeper networks")
print("• Study advanced architectures (ResNet, DenseNet concepts)")
print("• Learn about transfer learning and pre-trained models")
print("• Investigate automated hyperparameter optimization")
print("• Practice on domain-specific problems")

print("\\n" + "="*70)
print("🎉 MLP NOTEBOOK ENHANCEMENT COMPLETED SUCCESSFULLY! 🎉")
print("="*70)
print("This comprehensive guide covers everything from mathematical")
print("foundations to practical implementation and optimization.")
print("You now have a complete understanding of Multi-Layer")
print("Perceptrons and the tools to apply them effectively!")
print("="*70)

In [ ]:
# 🧠 1. ENHANCED MLP CLASS WITH TRAINING DYNAMICS TRACKING
print("🧠 1. Enhanced MLP Implementation with Training Monitoring")
print("-" * 60)

class EnhancedMLP:
    def __init__(self, layers, activation='relu', learning_rate=0.01, random_state=42):
        """
        Enhanced MLP with comprehensive training dynamics tracking
        
        Parameters:
        - layers: list of layer sizes [input_size, hidden1, hidden2, ..., output_size]
        - activation: activation function ('relu', 'sigmoid', 'tanh')
        - learning_rate: learning rate for gradient descent
        - random_state: random seed for reproducibility
        """
        np.random.seed(random_state)
        
        self.layers = layers
        self.learning_rate = learning_rate
        self.activation = activation
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        for i in range(len(layers) - 1):
            # Xavier/Glorot initialization
            limit = np.sqrt(6 / (layers[i] + layers[i + 1]))
            W = np.random.uniform(-limit, limit, (layers[i], layers[i + 1]))
            b = np.zeros((1, layers[i + 1]))
            
            self.weights.append(W)
            self.biases.append(b)
        
        # Training history tracking
        self.history = {
            'loss': [],
            'accuracy': [],
            'val_loss': [],
            'val_accuracy': [],
            'weights_evolution': [],
            'activations_evolution': [],
            'gradients_evolution': []
        }
        
        # Current forward pass storage
        self.activations = []
        self.z_values = []
    
    def _activation_function(self, x, derivative=False):
        """Apply activation function"""
        if self.activation == 'relu':
            if derivative:
                return (x > 0).astype(float)
            return np.maximum(0, x)
        elif self.activation == 'sigmoid':
            if derivative:
                s = self._activation_function(x)
                return s * (1 - s)
            return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        elif self.activation == 'tanh':
            if derivative:
                return 1 - np.tanh(x) ** 2
            return np.tanh(x)
    
    def _softmax(self, x):
        """Softmax activation for output layer"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def forward(self, X, store_activations=True):
        """Forward propagation with activation storage"""
        if store_activations:
            self.activations = [X]
            self.z_values = []
        
        current_input = X
        
        for i in range(len(self.weights)):
            z = np.dot(current_input, self.weights[i]) + self.biases[i]
            
            if store_activations:
                self.z_values.append(z)
            
            if i == len(self.weights) - 1:  # Output layer
                a = self._softmax(z)
            else:  # Hidden layers
                a = self._activation_function(z)
            
            if store_activations:
                self.activations.append(a)
            
            current_input = a
        
        return current_input
    
    def backward(self, X, y_true):
        """Backward propagation with gradient tracking"""
        m = X.shape[0]
        
        # Convert labels to one-hot if needed
        if len(y_true.shape) == 1:
            y_onehot = np.eye(self.layers[-1])[y_true]
        else:
            y_onehot = y_true
        
        # Calculate output error
        dA = self.activations[-1] - y_onehot
        
        gradients_w = []
        gradients_b = []
        
        # Backward through layers
        for i in reversed(range(len(self.weights))):
            # Calculate gradients
            dW = np.dot(self.activations[i].T, dA) / m
            db = np.sum(dA, axis=0, keepdims=True) / m
            
            gradients_w.insert(0, dW)
            gradients_b.insert(0, db)
            
            # Propagate error to previous layer
            if i > 0:
                dA = np.dot(dA, self.weights[i].T) * self._activation_function(self.z_values[i-1], derivative=True)
        
        # Update parameters
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * gradients_w[i]
            self.biases[i] -= self.learning_rate * gradients_b[i]
        
        return gradients_w, gradients_b
    
    def compute_loss(self, y_pred, y_true):
        """Compute cross-entropy loss"""
        if len(y_true.shape) == 1:
            y_onehot = np.eye(self.layers[-1])[y_true]
        else:
            y_onehot = y_true
        
        # Avoid log(0) by adding small epsilon
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        return -np.mean(np.sum(y_onehot * np.log(y_pred), axis=1))
    
    def predict(self, X):
        """Make predictions"""
        y_pred = self.forward(X, store_activations=False)
        return np.argmax(y_pred, axis=1)
    
    def fit(self, X_train, y_train, X_val=None, y_val=None, epochs=1000, 
            track_weights=True, verbose=True):
        """
        Train the MLP with comprehensive tracking
        """
        print(f"🏋️ Training MLP with architecture: {self.layers}")
        print(f"📊 Activation: {self.activation}, Learning Rate: {self.learning_rate}")
        print(f"🔢 Training samples: {X_train.shape[0]}, Features: {X_train.shape[1]}")
        print("-" * 50)
        
        for epoch in range(epochs):
            # Forward pass
            y_pred_train = self.forward(X_train)
            
            # Backward pass
            gradients_w, gradients_b = self.backward(X_train, y_train)
            
            # Calculate metrics
            train_loss = self.compute_loss(y_pred_train, y_train)
            train_pred = np.argmax(y_pred_train, axis=1)
            
            if len(y_train.shape) == 1:
                train_acc = accuracy_score(y_train, train_pred)
            else:
                train_acc = accuracy_score(np.argmax(y_train, axis=1), train_pred)
            
            # Store training metrics
            self.history['loss'].append(train_loss)
            self.history['accuracy'].append(train_acc)
            
            # Validation metrics
            if X_val is not None and y_val is not None:
                y_pred_val = self.forward(X_val, store_activations=False)
                val_loss = self.compute_loss(y_pred_val, y_val)
                val_pred = np.argmax(y_pred_val, axis=1)
                
                if len(y_val.shape) == 1:
                    val_acc = accuracy_score(y_val, val_pred)
                else:
                    val_acc = accuracy_score(np.argmax(y_val, axis=1), val_pred)
                
                self.history['val_loss'].append(val_loss)
                self.history['val_accuracy'].append(val_acc)
            
            # Track weights evolution (every 50 epochs to save memory)
            if track_weights and epoch % 50 == 0:
                weights_snapshot = [w.copy() for w in self.weights]
                self.history['weights_evolution'].append(weights_snapshot)
                
                # Track gradients magnitude
                grad_magnitudes = [np.mean(np.abs(grad)) for grad in gradients_w]
                self.history['gradients_evolution'].append(grad_magnitudes)
            
            # Print progress
            if verbose and epoch % 100 == 0:
                if X_val is not None:
                    print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
                else:
                    print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        
        print("\\n✅ Training completed!")
        return self.history

# Test the enhanced MLP on a simple dataset
print("\\n🔬 Testing Enhanced MLP on Binary Classification Dataset")
print("-" * 55)

# Create a challenging dataset
X_test, y_test = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                                   n_informative=2, n_clusters_per_class=2, 
                                   class_sep=0.8, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_test, y_test, test_size=0.3, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Create and train MLP
mlp_enhanced = EnhancedMLP(layers=[2, 10, 8, 2], activation='relu', learning_rate=0.01)
history = mlp_enhanced.fit(X_train, y_train, X_val, y_val, epochs=500, verbose=True)

# Make predictions
y_pred = mlp_enhanced.predict(X_val)
final_accuracy = accuracy_score(y_val, y_pred)
print(f"\\n🎯 Final Validation Accuracy: {final_accuracy:.4f}")

In [ ]:
# 🚀 ENHANCED MLP TRAINING DYNAMICS AND VISUALIZATION
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🎯 Enhanced MLP Analysis with Training Dynamics Visualization")
print("=" * 65)